# Paired validation M0–M1–M2: hai seed mới

Đây là notebook duy nhất cần chạy tiếp. Không chạy lại 07–09, không cần raw Amazon reviews.

1. Chọn GPU **T4**, ưu tiên runtime mới để không còn model cũ trên GPU.
2. Dữ liệu và summary từ các notebook trước phải còn trên Drive.
3. Chạy lần lượt các cell 1–7 bên dưới. Chỉ sửa `DRIVE_ROOT` nếu Drive trước đây dùng root khác.
4. Chờ `ALL_6_RUNS_SAVED`, export và gửi `paired_sampling_validation_bundle.zip`.

Hai training seed: **20270913**, **20280913**; sampler seed: **20271013**, **20281013**. Cùng seed, ba method dùng cùng khởi tạo/pair/negative, có checksum kiểm thực tế. Không đổi công thức, batch, epoch hoặc objective sau khi xem score.

**Resume:** lượt complete hợp lệ được bỏ qua; lượt đang dở phải chạy lại từ epoch 1, không có model checkpoint. Các lượt complete nằm trên Drive, không mất khi runtime ngắt. Không chạy đồng thời hai runtime trên cùng output folder.

**Giới hạn:** đây vẫn là 5 epoch/300 step trên validation, không phải final test. M2 đã được thiết kế sau khi xem s0. Resource s0 chỉ làm context vì runner mới bổ sung fingerprint instrumentation.


In [ ]:
# @title 1. Mount Drive và kiểm tra file có sẵn
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# CHỈ sửa dòng này nếu các notebook trước dùng thư mục Drive khác.
DRIVE_ROOT = Path('/content/drive/MyDrive/Phase2_Amazon_Audit')
GRAPH_DIR = DRIVE_ROOT / 'g2c_baby_p4'
MOSTPOP_PATH = DRIVE_ROOT / 'mostpop_validation' / 'mostpop_validation_summary.json'
BPR_PATH = DRIVE_ROOT / 'bpr_mf_sanity_v1' / 'bpr_mf_validation_summary.json'
FULL_LIGHTGCN_PATH = DRIVE_ROOT / 'full_lightgcn_sanity_v1' / 'full_lightgcn_validation_summary.json'
UNIFORM_PATH = DRIVE_ROOT / 'uniform_sampling_k65536_smoke_v1' / 'uniform_sampling_validation_summary.json'
DEGREE_PATH = DRIVE_ROOT / 'degree_aware_sampling_k65536_smoke_v1' / 'degree_aware_sampling_validation_summary.json'
FRONTIER_PATH = DRIVE_ROOT / 'frontier_normalized_sampling_k65536_smoke_v1' / 'frontier_normalized_sampling_validation_summary.json'
MANIFEST_PATH = GRAPH_DIR / 'baby_p4_g2c_manifest.json'
OUTPUT_DIR = DRIVE_ROOT / 'paired_sampling_validation_k65536_v1'

for required in (MANIFEST_PATH, MOSTPOP_PATH, BPR_PATH, FULL_LIGHTGCN_PATH,
                 UNIFORM_PATH, DEGREE_PATH, FRONTIER_PATH):
    if not required.is_file():
        raise FileNotFoundError(f'Thiếu artifact: {required}. Không chạy lại training; kiểm tra DRIVE_ROOT.')
print('INPUT_PATHS_READY')
print('Graph:', GRAPH_DIR)
print('Output riêng:', OUTPUT_DIR)


## Protocol và code đã khóa

Các cell dưới đây chứa engine đã chạy ở notebook 07–09, chỉ thêm audit khởi tạo/pair/negative. Không cần upload repo, config hoặc ZIP baseline riêng; notebook đọc summary có sẵn trên Drive.


In [ ]:
# @title 2. Protocol và engine đã khóa — KHÔNG SỬA
import json
import copy
import hashlib
import io
import uuid
import zipfile
import numpy as np

REGISTERED_PROTOCOL_SHA256 = "69de31ebe2a25da4b6000663f0dfee3ca0cd903dd2a767c6ac187174d86ec689"
PROTOCOL = json.loads("{\"protocol_id\":\"paired-sampling-validation-k65536-v1-2026-09-15\",\"registration_stage\":\"registered_before_two_additional_seed_runs_after_inspecting_s0\",\"matched_control_id\":\"static-controls-k65536-smoke-v1\",\"additional_seeds\":[{\"seed_id\":\"s1\",\"training_seed\":20270913,\"sampler_seed\":20271013,\"method_order\":[\"M0\",\"M1\",\"M2\"]},{\"seed_id\":\"s2\",\"training_seed\":20280913,\"sampler_seed\":20281013,\"method_order\":[\"M2\",\"M0\",\"M1\"]}],\"legacy_seed\":{\"seed_id\":\"s0\",\"training_seed\":20260913,\"sampler_seed\":20261013},\"legacy_summary_sha256\":{\"M0\":\"f29ed8f15d50a6d585ecb9cdcfc160232b64f9936e83fc8db6dac442bf941474\",\"M1\":\"46d83f7e6938c94be3376a5e71387e6a1bf9e9db32043d4dbbd10378f9bd07da\",\"M2\":\"26a1eb63db8d42582d955939623a0ef6a03082e22a1401b5255642d56021d0e3\"},\"methods\":{\"M0\":{\"method\":\"uniform\",\"source_notebook\":\"07_uniform_sampling_validation_sanity.ipynb\",\"source_notebook_file_sha256\":\"249feddd98083c10d2fe19ef6b7fac944397529af84d47c180b936194bfee547\",\"base_config_file_sha256\":\"b2b91b78262b562ad937be8a6328ebcd42f80b6a3b8e1695a988877e93c1b884\",\"base_config_semantic_sha256\":\"b2b91b78262b562ad937be8a6328ebcd42f80b6a3b8e1695a988877e93c1b884\",\"runner_source_sha256\":{\"training\":\"69f98cda02d52e614724d40c536e41d193d98e0c903ca73a326f153fa48ce79e\",\"evaluation\":\"7e9f670e9c23fc59892d49df2ff6bf43a5715bf0851dbe36ca642b4bf6656a21\"}},\"M1\":{\"method\":\"degree_proportional\",\"source_notebook\":\"08_degree_aware_sampling_validation_sanity.ipynb\",\"source_notebook_file_sha256\":\"a63a78fa14916f95c6ebe268cfdef31d48aaffc44dd2f3d81556a9b714bbe916\",\"base_config_file_sha256\":\"fdc9b95930b4975eb17f8e4ab5325bd835cb619f2891587ab419625c24812d9f\",\"base_config_semantic_sha256\":\"7d552ee2a536ab64346505008b69c0cef71343c215fe787b60065374f07f098c\",\"runner_source_sha256\":{\"training\":\"20345b01983f0232990aa0d35da6cc064430ea403707a8e567b8ec3bbb9cd517\",\"evaluation\":\"0f537b31c34e23b9abf789fe681819ab069bdcacb7779e5011c6ea5935c656e7\"}},\"M2\":{\"method\":\"frontier_normalized\",\"source_notebook\":\"09_frontier_normalized_sampling_validation_sanity.ipynb\",\"source_notebook_file_sha256\":\"fd06174d12067c7cb297a0f4b6792c9a70e4795f960e7c28997903955977ee46\",\"base_config_file_sha256\":\"b82651b29dd31a8eeaa279e2bc0c7612ee1daaa4c205f231dde39b65bef8563d\",\"base_config_semantic_sha256\":\"1a9f44cd020c0a99708e77ea672da07915e55b8af9d9946eeb238572992d1221\",\"runner_source_sha256\":{\"training\":\"a5b083a3c3b7ae6b12da1039be0dc043c187221d939563aebb846d5b6e5f4842\",\"evaluation\":\"d9e878c18828faea9bf3de29c2d3487ed8c57e727f11cbaab5ef5f72a7483a6b\"}}},\"shared_source_sha256\":{\"foundation\":\"e2a5ca5e0d8622161c3bd19064e4e12eb5cdebe919faa1832cd1f82f56487a96\",\"data\":\"69dac945c91e18ecd65a2ac49e7095a4e0465f7386de8a753fb8d8b848169426\"},\"source_hashes\":{\"train\":\"33abbd98b1d75b179bf45ef41ba3f5622d80005ad4aaf414320dbe48c17aeb72\",\"validation\":\"fd807f68fd03787931546ac31d7c2fe8e7ffcc7c43d3b92c627dce98fa87afca\"},\"expected_optimizer_steps\":300,\"expected_epochs\":5,\"required_gpu\":\"Tesla T4\",\"evaluation_split\":\"validation_only\",\"checkpoint_rule\":\"fixed_last_epoch\",\"resume_policy\":\"skip_only_checksum_verified_completed_runs; retain_incomplete_attempts; restart_incomplete_training_from_epoch_1; no_model_checkpoint\",\"primary_comparison\":\"paired M2-minus-M1 NDCG@20 under fixed optimizer/node budget\",\"secondary_comparisons\":[\"paired M2-minus-M0\",\"Recall@20\",\"coverage\",\"target-cohort hit transitions and rank diagnostics\",\"sampler/training time and peak training GPU memory\"],\"failure_diagnostics\":\"Additional rank thresholds 100 and 1000 are descriptive only, not substituted for the registered top-20 objective.\",\"claim_boundary\":\"Hai seed validation bổ sung ở cùng 5 epoch/300 optimizer step. M2 giữ nguyên công thức đã chọn sau khi xem s0; đây là kiểm chứng sau thiết kế, không phải holdout độc lập của quá trình chọn phương pháp. Báo cáo từng seed và quality mean/sample SD, không chọn best seed, không significance claim, không đổi objective, không đọc test và không tự promote M2. Ba seed không chứng minh hiệu quả ở budget huấn luyện khác hoặc dataset khác.\",\"resource_boundary\":\"Training wall time gồm các integrity/trace audit, fingerprint đầu vào mỗi epoch và CPU sampler; không gồm đọc dữ liệu, checksum embedding ban đầu, validation hoặc lưu kết quả. Sampled training peak GPU được reset riêng trước training, full-graph validation peak reset riêng sau khi dựng inference graph. So sánh resource ưu tiên paired run mới trong cùng seed/môi trường. Resource s0 chỉ là legacy context, không gộp mean/SD với run mới vì có instrumentation bổ sung; quality có thể tổng hợp ba seed nếu core/data/evaluator và các runtime field đã ghi giữ nguyên. Process peak RSS là lifetime high-water mark, không phải peak CPU riêng của từng run.\",\"additional_run_environment_rule\":\"All additional runs must match Python/Torch/CUDA/GPU/NumPy/SciPy environment fingerprint; resume rejects a different runtime fingerprint.\",\"legacy_quality_environment_policy\":\"Include s0 in quality mean/sample SD only when its recorded Python/Torch/CUDA/GPU fields match all additional runs. Otherwise display s0 separately; no mixed-runtime pooled quality statistics.\",\"completion_publication\":\"Write/close/verify completion marker inside a unique attempt directory, then same-filesystem rename to COMPLETED.json; one runtime per output folder; no concurrent writers.\"}")
BASE_CONFIGS = json.loads("{\"M0\":{\"claim_boundary\":\"One fixed validation-only M0 uniform sampled-training smoke run at the predeclared k_l=65536 layer budget. This is an implementation and resource-envelope probe, not the middle point of a tested budget grid. No test access, hyperparameter search, method selection, learned sampler, sampled-inference claim, or comparison to full LightGCN as a budget-matched method.\",\"config_id\":\"uniform-sampling-k65536-smoke-v1-2026-09-14\",\"evaluation\":{\"candidate_universe\":\"all frozen training items minus strict prior mapped positives\",\"eval_batch_size\":128,\"inference_graph\":\"full_frozen_training_graph\",\"k\":20,\"rank_tie_break\":\"item_idx_ascending\",\"split\":\"validation_only\",\"target_unit\":\"one relevant item per target row\",\"timestamp_rule\":\"events at the target timestamp are not prior history\"},\"matched_control_id\":\"static-controls-k65536-smoke-v1\",\"model\":{\"aggregation\":\"unweighted_mean_of_ego_and_all_layers\",\"bias\":false,\"embedding_dim\":64,\"initialization_std\":0.01,\"layers\":3,\"name\":\"Sampled-LightGCN\",\"recommender_self_loops\":false},\"sampler\":{\"candidate_rule\":\"unique_neighbors_of_previous_K_excluding_previous_K\",\"cross_layer_reentry\":true,\"exact_k_without_replacement\":true,\"k_l\":[65536,65536,65536],\"k_l_interpretation\":\"global_context_node_budget_per_batch_and_layer_equal_to_training_batch_size_for_this_smoke_only\",\"method\":\"uniform\",\"normalization\":\"rectangular_sampled_local_bi_normalization\",\"positive_edge_policy\":\"retain\",\"seed\":20261013,\"state_rule\":\"K_l_equals_V0_union_V_l_not_cumulative_union\"},\"training\":{\"batch_size\":65536,\"epoch_selection\":\"fixed_last_epoch\",\"epochs\":5,\"l2_coefficient\":0.000001,\"learning_rate\":0.01,\"mixed_precision\":false,\"negative_sampling\":\"uniform_training_catalog_reject_all_training_positives\",\"optimizer\":\"SparseAdam\",\"seed\":20260913,\"training_pair_order\":\"seeded_epoch_permutation\"}},\"M1\":{\"claim_boundary\":\"One fixed validation-only M1 training-degree-aware sampled-training smoke run at the predeclared k_l=65536 layer budget. This is an implementation and resource-envelope probe, not the middle point of a tested budget grid. No test access, hyperparameter search, method selection, learned sampler, sampled-inference claim, or comparison to full LightGCN as a budget-matched method.\",\"config_id\":\"degree-aware-sampling-k65536-smoke-v1-2026-09-14\",\"evaluation\":{\"candidate_universe\":\"all frozen training items minus strict prior mapped positives\",\"eval_batch_size\":128,\"inference_graph\":\"full_frozen_training_graph\",\"k\":20,\"rank_tie_break\":\"item_idx_ascending\",\"split\":\"validation_only\",\"target_unit\":\"one relevant item per target row\",\"timestamp_rule\":\"events at the target timestamp are not prior history\"},\"matched_control_id\":\"static-controls-k65536-smoke-v1\",\"model\":{\"aggregation\":\"unweighted_mean_of_ego_and_all_layers\",\"bias\":false,\"embedding_dim\":64,\"initialization_std\":0.01,\"layers\":3,\"name\":\"Sampled-LightGCN\",\"recommender_self_loops\":false},\"sampler\":{\"candidate_rule\":\"unique_neighbors_of_previous_K_excluding_previous_K\",\"cross_layer_reentry\":true,\"degree_exponent\":1,\"degree_source\":\"full_frozen_training_graph_only\",\"exact_k_without_replacement\":true,\"k_l\":[65536,65536,65536],\"k_l_interpretation\":\"global_context_node_budget_per_batch_and_layer_equal_to_training_batch_size_for_this_smoke_only\",\"method\":\"degree_proportional\",\"normalization\":\"rectangular_sampled_local_bi_normalization\",\"positive_edge_policy\":\"retain\",\"seed\":20261013,\"state_rule\":\"K_l_equals_V0_union_V_l_not_cumulative_union\"},\"training\":{\"batch_size\":65536,\"epoch_selection\":\"fixed_last_epoch\",\"epochs\":5,\"l2_coefficient\":0.000001,\"learning_rate\":0.01,\"mixed_precision\":false,\"negative_sampling\":\"uniform_training_catalog_reject_all_training_positives\",\"optimizer\":\"SparseAdam\",\"seed\":20260913,\"training_pair_order\":\"seeded_epoch_permutation\"}},\"M2\":{\"claim_boundary\":\"One fixed validation-only M2 frontier-normalized sampled-training smoke run at the predeclared k_l=65536 layer budget. This project-candidate probe tests one formula selected after reading the M0-M1 validation controls. No test access, hyperparameter search, learned policy, sampled-inference claim, or comparison to full LightGCN as a budget-matched method.\",\"config_id\":\"frontier-normalized-sampling-k65536-smoke-v1-2026-09-15\",\"evaluation\":{\"candidate_universe\":\"all frozen training items minus strict prior mapped positives\",\"eval_batch_size\":128,\"inference_graph\":\"full_frozen_training_graph\",\"k\":20,\"rank_tie_break\":\"item_idx_ascending\",\"split\":\"validation_only\",\"target_unit\":\"one relevant item per target row\",\"timestamp_rule\":\"events at the target timestamp are not prior history\"},\"matched_control_id\":\"static-controls-k65536-smoke-v1\",\"method_role\":\"single_project_candidate_after_static_controls\",\"model\":{\"aggregation\":\"unweighted_mean_of_ego_and_all_layers\",\"bias\":false,\"embedding_dim\":64,\"initialization_std\":0.01,\"layers\":3,\"name\":\"Sampled-LightGCN\",\"recommender_self_loops\":false},\"sampler\":{\"candidate_rule\":\"unique_neighbors_of_previous_K_excluding_previous_K\",\"cross_layer_reentry\":true,\"degree_exponent\":-0.5,\"degree_source\":\"full_frozen_training_graph_only\",\"exact_k_without_replacement\":true,\"frontier_support_source\":\"count_training_edges_from_candidate_to_previous_K\",\"k_l\":[65536,65536,65536],\"k_l_interpretation\":\"global_context_node_budget_per_batch_and_layer_equal_to_training_batch_size_for_this_smoke_only\",\"method\":\"frontier_normalized\",\"normalization\":\"rectangular_sampled_local_bi_normalization\",\"positive_edge_policy\":\"retain\",\"priority_formula\":\"log(frontier_support) - 0.5*log(training_degree) + gumbel\",\"seed\":20261013,\"state_rule\":\"K_l_equals_V0_union_V_l_not_cumulative_union\",\"support_exponent\":1},\"training\":{\"batch_size\":65536,\"epoch_selection\":\"fixed_last_epoch\",\"epochs\":5,\"l2_coefficient\":0.000001,\"learning_rate\":0.01,\"mixed_precision\":false,\"negative_sampling\":\"uniform_training_catalog_reject_all_training_positives\",\"optimizer\":\"SparseAdam\",\"seed\":20260913,\"training_pair_order\":\"seeded_epoch_permutation\"}}}")
FOUNDATION_SOURCE = "from collections import Counter, defaultdict\nfrom dataclasses import dataclass\nfrom itertools import groupby\nimport csv\nimport gc\nimport gzip\nimport hashlib\nimport json\nimport math\nimport platform\nimport resource\nimport time\n\nimport numpy as np\nimport scipy.sparse as sp\n\nMOSTPOP_SUMMARY_SHA256 = '00332090cae73a563a7fcafde895572fb06d574366ce37bf6f9d90208dfccf62'\nBPR_SUMMARY_SHA256 = '611bd820c66c57ef7a06d18e52a36a61942bfaf55fc9a615ee6b4708ac5a1f74'\nFULL_LIGHTGCN_SUMMARY_SHA256 = 'ebddb3ece87abc0098106dd2271b9134f641ca8e2582e35491b9f26b6fe0faa9'\nUNIFORM_SUMMARY_SHA256 = 'f29ed8f15d50a6d585ecb9cdcfc160232b64f9936e83fc8db6dac442bf941474'\nDEGREE_SUMMARY_SHA256 = '46d83f7e6938c94be3376a5e71387e6a1bf9e9db32043d4dbbd10378f9bd07da'\n\n\ndef sha256_file(path, chunk_size=1024 * 1024):\n    digest = hashlib.sha256()\n    with path.open('rb') as handle:\n        while True:\n            chunk = handle.read(chunk_size)\n            if not chunk:\n                break\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef resolve_artifact(entry, fallback_name):\n    recorded = Path(entry['path'])\n    fallback = GRAPH_DIR / fallback_name\n    if recorded.exists():\n        return recorded\n    if fallback.exists():\n        return fallback\n    raise FileNotFoundError(f'Không tìm thấy artifact: {recorded} hoặc {fallback}')\n\n\ndef row_metrics(ranks, k):\n    ranks = np.asarray(ranks, dtype=np.int64)\n    if ranks.size == 0:\n        return {'rows': 0, 'hits_at_k': 0, 'recall_at_k': None, 'ndcg_at_k': None, 'k': k}\n    hits = ranks <= k\n    discounts = np.zeros(ranks.size, dtype=np.float64)\n    discounts[hits] = 1.0 / np.log2(ranks[hits] + 1.0)\n    return {\n        'rows': int(ranks.size),\n        'hits_at_k': int(hits.sum()),\n        'recall_at_k': float(hits.mean()),\n        'ndcg_at_k': float(discounts.mean()),\n        'k': int(k),\n    }\n\n\ndef item_cohort(degree):\n    if degree >= 397:\n        return 'head'\n    if degree >= 13:\n        return 'body'\n    return 'tail'\n\n\ndef user_cohort(degree):\n    if degree == 1:\n        return 'singleton'\n    if degree <= 3:\n        return 'repeat_light'\n    return 'active'\n\n\ndef grouped_metrics(ranks, labels, k):\n    ranks = np.asarray(ranks)\n    labels = np.asarray(labels)\n    result = {}\n    for label in sorted(set(labels.tolist())):\n        mask = labels == label\n        result[label] = {**row_metrics(ranks[mask], k), 'target_share': float(mask.mean())}\n    return result\n\n\ndef process_peak_rss_mb():\n    return float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0)\n\n\ndef positive_collision_mask(users, items, positive_keys, item_count):\n    keys = users.astype(np.int64, copy=False) * item_count + items.astype(np.int64, copy=False)\n    positions = np.searchsorted(positive_keys, keys)\n    in_bounds = positions < positive_keys.size\n    collisions = np.zeros(keys.size, dtype=bool)\n    collisions[in_bounds] = positive_keys[positions[in_bounds]] == keys[in_bounds]\n    return collisions\n\n\ndef sample_exact_uniform_negatives(users, rng, positive_keys, item_count):\n    negatives = rng.integers(0, item_count, size=users.size, dtype=np.int32)\n    collisions = positive_collision_mask(users, negatives, positive_keys, item_count)\n    resampled = 0\n    rounds = 0\n    while collisions.any():\n        count = int(collisions.sum())\n        negatives[collisions] = rng.integers(0, item_count, size=count, dtype=np.int32)\n        resampled += count\n        rounds += 1\n        if rounds > 100:\n            raise RuntimeError('Negative rejection sampler không hội tụ')\n        collisions = positive_collision_mask(users, negatives, positive_keys, item_count)\n    return negatives, resampled\n"
DATA_SOURCE = "manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))\nmostpop = json.loads(MOSTPOP_PATH.read_text(encoding='utf-8'))\nbpr = json.loads(BPR_PATH.read_text(encoding='utf-8'))\nfull_lightgcn = json.loads(FULL_LIGHTGCN_PATH.read_text(encoding='utf-8'))\nuniform_m0 = json.loads(UNIFORM_PATH.read_text(encoding='utf-8'))\ndegree_m1 = json.loads(DEGREE_PATH.read_text(encoding='utf-8'))\n\nuniform_config = {key: value for key, value in uniform_m0['registered_config'].items() if key != 'file_sha256'}\ndegree_config = {key: value for key, value in degree_m1['registered_config'].items() if key != 'file_sha256'}\nmatched_sampler_fields = (\n    'candidate_rule', 'cross_layer_reentry', 'exact_k_without_replacement', 'k_l',\n    'k_l_interpretation', 'normalization', 'positive_edge_policy', 'seed', 'state_rule',\n)\nmatched_sampler_core_equal = all(\n    uniform_config['sampler'][field] == degree_config['sampler'][field] == CONFIG['sampler'][field]\n    for field in matched_sampler_fields\n)\ncontrol_checks = {\n    'mostpop_sha256_matches': sha256_file(MOSTPOP_PATH) == MOSTPOP_SUMMARY_SHA256,\n    'bpr_sha256_matches': sha256_file(BPR_PATH) == BPR_SUMMARY_SHA256,\n    'full_lightgcn_sha256_matches': sha256_file(FULL_LIGHTGCN_PATH) == FULL_LIGHTGCN_SUMMARY_SHA256,\n    'uniform_sha256_matches': sha256_file(UNIFORM_PATH) == UNIFORM_SUMMARY_SHA256,\n    'degree_sha256_matches': sha256_file(DEGREE_PATH) == DEGREE_SUMMARY_SHA256,\n    'mostpop_integrity_passed': all(mostpop['integrity_assertions'].values()),\n    'bpr_integrity_passed': all(bpr['integrity_assertions'].values()),\n    'full_lightgcn_integrity_passed': all(full_lightgcn['integrity_assertions'].values()),\n    'uniform_integrity_passed': all(uniform_m0['integrity_assertions'].values()),\n    'degree_integrity_passed': all(degree_m1['integrity_assertions'].values()),\n    'uniform_degree_frontier_matched_control_id': uniform_config['matched_control_id'] == degree_config['matched_control_id'] == CONFIG['matched_control_id'],\n    'matched_model_equal': uniform_config['model'] == degree_config['model'] == CONFIG['model'],\n    'matched_training_equal': uniform_config['training'] == degree_config['training'] == CONFIG['training'],\n    'matched_evaluation_equal': uniform_config['evaluation'] == degree_config['evaluation'] == CONFIG['evaluation'],\n    'matched_sampler_core_equal_except_proposal': matched_sampler_core_equal,\n    'uniform_method_is_uniform': uniform_config['sampler']['method'] == 'uniform',\n    'degree_method_is_degree_proportional': degree_config['sampler']['method'] == 'degree_proportional',\n    'frontier_method_is_frontier_normalized': CONFIG['sampler']['method'] == 'frontier_normalized',\n}\nif not all(control_checks.values()):\n    raise AssertionError(control_checks)\n\nuser_count = int(manifest['training_graph']['users'])\nitem_count = int(manifest['training_graph']['items'])\nnode_count = user_count + item_count\ntrain_entry = manifest['artifacts']['train_edges']\nvalidation_entry = manifest['artifacts']['validation_targets']\ntrain_path = resolve_artifact(train_entry, 'baby_p4_train_edges.csv.gz')\nvalidation_path = resolve_artifact(validation_entry, 'baby_p4_validation_targets.csv.gz')\n\nprint('1/7 Checksum...')\ntrain_sha = sha256_file(train_path)\nvalidation_sha = sha256_file(validation_path)\nif train_sha != train_entry['sha256'] or validation_sha != validation_entry['sha256']:\n    raise AssertionError('Checksum data không khớp manifest')\n\nvalidation_rows_expected = int(validation_entry['rows'])\ntarget_users = np.empty(validation_rows_expected, dtype=np.int32)\ntarget_items = np.empty(validation_rows_expected, dtype=np.int32)\ntarget_timestamps = np.empty(validation_rows_expected, dtype=np.int64)\ntarget_source_rows = np.empty(validation_rows_expected, dtype=np.int64)\ntarget_candidate_counts = np.empty(validation_rows_expected, dtype=np.int32)\n\nprint('2/7 Đọc validation target...')\nvalidation_rows = 0\nwith gzip.open(validation_path, 'rt', encoding='utf-8', newline='') as handle:\n    for index, row in enumerate(csv.DictReader(handle)):\n        if index >= validation_rows_expected:\n            raise AssertionError('Validation artifact có nhiều row hơn manifest')\n        target_users[index] = int(row['user_idx'])\n        target_items[index] = int(row['item_idx'])\n        target_timestamps[index] = int(row['timestamp_ms'])\n        target_source_rows[index] = int(row['source_row'])\n        target_candidate_counts[index] = int(row['candidate_count'])\n        validation_rows = index + 1\nif validation_rows != validation_rows_expected:\n    raise AssertionError('Validation row count không khớp manifest')\n\neval_user_mask = np.zeros(user_count, dtype=bool)\neval_user_mask[np.unique(target_users)] = True\nbase_history = defaultdict(list)\ntrain_rows_expected = int(train_entry['rows'])\ntrain_users = np.empty(train_rows_expected, dtype=np.int32)\ntrain_items = np.empty(train_rows_expected, dtype=np.int32)\n\nprint('3/7 Đọc training edge...')\ntrain_rows = 0\nwith gzip.open(train_path, 'rt', encoding='utf-8', newline='') as handle:\n    for index, row in enumerate(csv.DictReader(handle)):\n        if index >= train_rows_expected:\n            raise AssertionError('Training artifact có nhiều row hơn manifest')\n        user_idx = int(row['user_idx'])\n        item_idx = int(row['item_idx'])\n        train_users[index] = user_idx\n        train_items[index] = item_idx\n        if eval_user_mask[user_idx]:\n            base_history[user_idx].append(item_idx)\n        train_rows = index + 1\nif train_rows != train_rows_expected:\n    raise AssertionError('Training row count không khớp manifest')\n\nprint('4/7 Degree, positive keys và strict prior history...')\nuser_degrees = np.bincount(train_users, minlength=user_count).astype(np.int64)\nitem_degrees = np.bincount(train_items, minlength=item_count).astype(np.int64)\nnode_degrees = np.concatenate((user_degrees, item_degrees))\nif int(user_degrees.sum()) != train_rows or int(item_degrees.sum()) != train_rows:\n    raise AssertionError('Degree mass không bằng training rows')\n\npositive_keys = train_users.astype(np.int64) * item_count + train_items.astype(np.int64)\npositive_keys.sort()\nif np.any(np.diff(positive_keys) == 0):\n    raise AssertionError('Training user-item pair không unique')\n\ntargets_by_user = defaultdict(list)\nfor target_index, user_idx in enumerate(target_users):\n    targets_by_user[int(user_idx)].append(target_index)\nhistories = [None] * validation_rows\ntarget_not_in_prior = True\ncandidate_checks = 0\nfor user_idx, indices in targets_by_user.items():\n    prior = set(base_history[user_idx])\n    indices.sort(key=lambda index: (int(target_timestamps[index]), int(target_source_rows[index])))\n    for _, timestamp_group in groupby(indices, key=lambda index: int(target_timestamps[index])):\n        group = list(timestamp_group)\n        for target_index in group:\n            target_item = int(target_items[target_index])\n            if target_item in prior:\n                target_not_in_prior = False\n                raise AssertionError('Validation target đã nằm trong strict prior history')\n            if item_count - len(prior) != int(target_candidate_counts[target_index]):\n                raise AssertionError('Candidate count không khớp strict prior history')\n            histories[target_index] = np.asarray(sorted(prior), dtype=np.int32)\n            candidate_checks += 1\n        prior.update(int(target_items[index]) for index in group)\nif any(history is None for history in histories):\n    raise AssertionError('Có validation target thiếu history')\n\nprint('5/7 Negative sampler preflight...')\npreflight_rng = np.random.default_rng(CONFIG['training']['seed'])\npreflight_indices = preflight_rng.choice(train_rows, size=min(50000, train_rows), replace=False)\npreflight_users = train_users[preflight_indices]\npreflight_negatives, preflight_resamples = sample_exact_uniform_negatives(\n    preflight_users, preflight_rng, positive_keys, item_count\n)\nif positive_collision_mask(preflight_users, preflight_negatives, positive_keys, item_count).any():\n    raise AssertionError('Negative preflight có positive collision')\n\nprint('6/7 Dựng training-only symmetric CSR có edge ID...')\nedge_ids = np.arange(train_rows, dtype=np.int64) + 1\nglobal_items = user_count + train_items.astype(np.int64)\ncsr_rows = np.concatenate((train_users.astype(np.int64), global_items))\ncsr_cols = np.concatenate((global_items, train_users.astype(np.int64)))\ncsr_data = np.concatenate((edge_ids, edge_ids))\ngraph_csr = sp.csr_matrix((csr_data, (csr_rows, csr_cols)), shape=(node_count, node_count))\ngraph_csr.sort_indices()\nif graph_csr.nnz != 2 * train_rows:\n    raise AssertionError('CSR không có đúng hai direction cho mỗi training edge')\n\nsource_assertions = {\n    **control_checks,\n    'train_sha256_matches_manifest': train_sha == train_entry['sha256'],\n    'validation_sha256_matches_manifest': validation_sha == validation_entry['sha256'],\n    'train_rows_match_manifest': train_rows == int(train_entry['rows']),\n    'validation_rows_match_manifest': validation_rows == int(validation_entry['rows']),\n    'degree_mass_matches_train_rows': int(user_degrees.sum()) == train_rows == int(item_degrees.sum()),\n    'training_pairs_unique': not np.any(np.diff(positive_keys) == 0),\n    'candidate_count_matches_every_target': candidate_checks == validation_rows,\n    'target_not_in_strict_prior_history': target_not_in_prior,\n    'negative_sampler_preflight_has_no_positive_collision': True,\n    'training_csr_has_two_directions_per_edge': graph_csr.nnz == 2 * train_rows,\n    'test_targets_not_read': True,\n}\nif not all(source_assertions.values()):\n    raise AssertionError(source_assertions)\nprint('7/7 Source gate pass.')\n"
ENGINES = json.loads("{\"M0\":{\"training\":\"import torch\\nfrom torch import nn\\nimport torch.nn.functional as F\\n\\nif not torch.cuda.is_available():\\n    raise RuntimeError('Uniform sampling smoke yêu cầu Colab GPU CUDA.')\\ndevice = torch.device('cuda')\\ntorch.backends.cuda.matmul.allow_tf32 = False\\n\\n\\n@dataclass\\nclass NumpyBlock:\\n    source_nodes: np.ndarray\\n    target_nodes: np.ndarray\\n    source_local: np.ndarray\\n    target_local: np.ndarray\\n    weights: np.ndarray\\n    edge_ids: np.ndarray\\n\\n\\ndef candidate_nodes(previous_nodes):\\n    rows = graph_csr[previous_nodes]\\n    neighbors = np.unique(rows.indices.astype(np.int64, copy=False))\\n    positions = np.searchsorted(previous_nodes, neighbors)\\n    in_previous = np.zeros(neighbors.size, dtype=bool)\\n    bounded = positions < previous_nodes.size\\n    in_previous[bounded] = previous_nodes[positions[bounded]] == neighbors[bounded]\\n    return neighbors[~in_previous]\\n\\n\\ndef uniform_exact_k(candidates, requested_k, rng):\\n    effective_k = min(int(requested_k), int(candidates.size))\\n    if effective_k == 0:\\n        return np.empty(0, dtype=np.int64)\\n    uniforms = np.clip(rng.random(candidates.size), np.finfo(np.float64).tiny, 1.0 - np.finfo(np.float64).eps)\\n    gumbels = -np.log(-np.log(uniforms))\\n    if effective_k == candidates.size:\\n        selected = candidates.copy()\\n    else:\\n        threshold = candidates.size - effective_k\\n        selected = candidates[np.argpartition(gumbels, threshold)[threshold:]]\\n    selected.sort()\\n    return selected\\n\\n\\ndef build_block(source_nodes, target_nodes):\\n    sliced = graph_csr[target_nodes]\\n    source_global = sliced.indices.astype(np.int64, copy=False)\\n    target_local_all = np.repeat(np.arange(target_nodes.size, dtype=np.int64), np.diff(sliced.indptr))\\n    positions = np.searchsorted(source_nodes, source_global)\\n    keep = positions < source_nodes.size\\n    valid_positions = positions[keep]\\n    valid_globals = source_global[keep]\\n    matched = source_nodes[valid_positions] == valid_globals\\n    keep_indices = np.flatnonzero(keep)[matched]\\n    source_local = positions[keep_indices].astype(np.int64, copy=False)\\n    target_local = target_local_all[keep_indices].astype(np.int64, copy=False)\\n    underlying_edge_ids = sliced.data[keep_indices].astype(np.int64, copy=False) - 1\\n    if source_local.size == 0:\\n        weights = np.empty(0, dtype=np.float32)\\n    else:\\n        source_degree = np.bincount(source_local, minlength=source_nodes.size)\\n        target_degree = np.bincount(target_local, minlength=target_nodes.size)\\n        weights = (1.0 / np.sqrt(source_degree[source_local] * target_degree[target_local])).astype(np.float32)\\n    return NumpyBlock(source_nodes, target_nodes, source_local, target_local, weights, underlying_edge_ids)\\n\\n\\ndef build_trace(v0, rng):\\n    k_sets = [v0]\\n    blocks = []\\n    layers = []\\n    previous = v0\\n    for layer_index, requested_k in enumerate(CONFIG['sampler']['k_l'], start=1):\\n        candidates = candidate_nodes(previous)\\n        selected = uniform_exact_k(candidates, requested_k, rng)\\n        current = np.union1d(v0, selected)\\n        block = build_block(current, previous)\\n        layers.append({\\n            'layer': layer_index,\\n            'candidate_nodes': candidates,\\n            'selected_nodes': selected,\\n            'requested_k': int(requested_k),\\n        })\\n        k_sets.append(current)\\n        blocks.append(block)\\n        previous = current\\n    return {'v0': v0, 'k_sets': k_sets, 'blocks': blocks, 'layers': layers}\\n\\n\\ndef trace_fingerprint(trace):\\n    digest = hashlib.sha256()\\n    for layer, block in zip(trace['layers'], trace['blocks']):\\n        for array in (layer['candidate_nodes'], layer['selected_nodes'], block.edge_ids):\\n            digest.update(np.asarray(array, dtype=np.int64).tobytes())\\n    return digest.hexdigest()\\n\\n\\ndef trace_contract(trace):\\n    exact_k = True\\n    unique = True\\n    disjoint = True\\n    state_rule = True\\n    original_edges = True\\n    no_self_loop = True\\n    finite_weights = True\\n    for index, (layer, block) in enumerate(zip(trace['layers'], trace['blocks']), start=1):\\n        candidates = layer['candidate_nodes']\\n        selected = layer['selected_nodes']\\n        previous = trace['k_sets'][index - 1]\\n        expected_current = np.union1d(trace['v0'], selected)\\n        exact_k &= selected.size == min(layer['requested_k'], candidates.size)\\n        unique &= np.unique(selected).size == selected.size\\n        disjoint &= np.intersect1d(selected, previous).size == 0\\n        state_rule &= np.array_equal(trace['k_sets'][index], expected_current)\\n        original_edges &= bool(np.all((block.edge_ids >= 0) & (block.edge_ids < train_rows)))\\n        if block.source_local.size:\\n            no_self_loop &= not np.any(block.source_nodes[block.source_local] == block.target_nodes[block.target_local])\\n        finite_weights &= bool(np.isfinite(block.weights).all())\\n    return {\\n        'exact_k_every_layer': bool(exact_k),\\n        'sampled_nodes_unique': bool(unique),\\n        'sampled_nodes_disjoint_from_previous_K': bool(disjoint),\\n        'K_l_equals_V0_union_V_l': bool(state_rule),\\n        'sampled_blocks_use_only_registered_training_edges': bool(original_edges),\\n        'sampled_blocks_have_no_self_loop': bool(no_self_loop),\\n        'sampled_block_weights_finite': bool(finite_weights),\\n    }\\n\\n\\ndef block_to_torch(block):\\n    indices = torch.from_numpy(np.stack((block.target_local, block.source_local))).to(device=device, dtype=torch.long)\\n    values = torch.from_numpy(block.weights).to(device=device)\\n    return torch.sparse_coo_tensor(\\n        indices, values,\\n        size=(block.target_nodes.size, block.source_nodes.size),\\n        device=device,\\n        is_coalesced=False,\\n    ).coalesce()\\n\\n\\ndef verify_rectangular_direction_oracle():\\n    indices = torch.tensor([[0, 1], [0, 1]], device=device)\\n    values = torch.tensor([1.0, 1.0], device=device)\\n    block = torch.sparse_coo_tensor(indices, values, (2, 2), device=device).coalesce()\\n    source = torch.tensor([[3.0], [7.0]], device=device)\\n    torch.testing.assert_close(torch.sparse.mm(block, source), source)\\n\\n\\nverify_rectangular_direction_oracle()\\n\\n\\nclass SampledLightGCN(nn.Module):\\n    def __init__(self, nodes, embedding_dim, layers, init_std):\\n        super().__init__()\\n        self.embedding = nn.Embedding(nodes, embedding_dim, sparse=True)\\n        self.layers = int(layers)\\n        nn.init.normal_(self.embedding.weight, std=init_std)\\n\\n    def sampled_propagate(self, trace):\\n        torch_blocks = [block_to_torch(block) for block in trace['blocks']]\\n        depth_outputs = [self.embedding(torch.from_numpy(trace['v0']).to(device=device, dtype=torch.long))]\\n        for depth in range(1, self.layers + 1):\\n            current_nodes = trace['k_sets'][depth]\\n            hidden = self.embedding(torch.from_numpy(current_nodes).to(device=device, dtype=torch.long))\\n            for layer_index in range(depth - 1, -1, -1):\\n                hidden = torch.sparse.mm(torch_blocks[layer_index], hidden)\\n            depth_outputs.append(hidden)\\n        return torch.stack(depth_outputs, dim=0).mean(dim=0)\\n\\n    def full_propagate(self, normalized_adjacency):\\n        current = self.embedding.weight\\n        combined = current / (self.layers + 1)\\n        for _ in range(self.layers):\\n            current = torch.sparse.mm(normalized_adjacency, current)\\n            combined = combined + current / (self.layers + 1)\\n        return combined\\n\\n\\ndef endpoint_locations(v0, users, positives, negatives):\\n    global_users = users.astype(np.int64, copy=False)\\n    global_positives = user_count + positives.astype(np.int64, copy=False)\\n    global_negatives = user_count + negatives.astype(np.int64, copy=False)\\n    locations = [np.searchsorted(v0, values) for values in (global_users, global_positives, global_negatives)]\\n    for values, found in zip((global_users, global_positives, global_negatives), locations):\\n        if not np.array_equal(v0[found], values):\\n            raise AssertionError('Triplet endpoint không nằm trong V0')\\n    return locations\\n\\n\\ndef update_trace_accounting(trace, selected_seen, computation_seen, edge_seen, layer_totals, selected_item_slots):\\n    computation_seen[trace['v0']] = True\\n    for index, (layer, block) in enumerate(zip(trace['layers'], trace['blocks'])):\\n        selected = layer['selected_nodes']\\n        selected_seen[selected] = True\\n        computation_seen[trace['k_sets'][index + 1]] = True\\n        edge_seen[block.edge_ids] = True\\n        layer_totals[index]['candidate_node_slots'] += int(layer['candidate_nodes'].size)\\n        layer_totals[index]['selected_context_node_slots'] += int(selected.size)\\n        layer_totals[index]['directed_block_entries'] += int(block.edge_ids.size)\\n        selected_items = selected[selected >= user_count] - user_count\\n        if selected_items.size:\\n            degrees = item_degrees[selected_items]\\n            selected_item_slots['head'] += int((degrees >= 397).sum())\\n            selected_item_slots['body'] += int(((degrees >= 13) & (degrees <= 396)).sum())\\n            selected_item_slots['tail'] += int((degrees <= 12).sum())\\n\\n\\ntorch.manual_seed(CONFIG['training']['seed'])\\ntorch.cuda.manual_seed_all(CONFIG['training']['seed'])\\nmodel = SampledLightGCN(\\n    node_count,\\n    CONFIG['model']['embedding_dim'],\\n    CONFIG['model']['layers'],\\n    CONFIG['model']['initialization_std'],\\n).to(device)\\ninitial_probe = model.embedding.weight[:1024].detach().cpu().clone()\\ninitial_embedding_sha256 = embedding_sha256(model.embedding.weight)\\nepoch_input_fingerprints = []\\noptimizer = torch.optim.SparseAdam(model.parameters(), lr=CONFIG['training']['learning_rate'])\\n\\ntorch.cuda.reset_peak_memory_stats(device)\\ntraining_started = time.perf_counter()\\nepoch_records = []\\noptimizer_steps = 0\\ntotal_negative_resamples = 0\\ndeterministic_replay_passed = False\\nall_trace_contracts = []\\n\\nfor epoch in range(CONFIG['training']['epochs']):\\n    epoch_started = time.perf_counter()\\n    pair_rng = np.random.default_rng(CONFIG['training']['seed'] + epoch)\\n    order = pair_rng.permutation(train_rows)\\n    negatives_all, resampled = sample_exact_uniform_negatives(train_users, pair_rng, positive_keys, item_count)\\n    total_negative_resamples += resampled\\n    fingerprint_started = time.perf_counter()\\n    epoch_input_fingerprints.append({\\n        'epoch': epoch + 1,\\n        'pair_order_sha256': array_sha256(order),\\n        'negative_items_sha256': array_sha256(negatives_all),\\n    })\\n    fingerprint_seconds = time.perf_counter() - fingerprint_started\\n    sampler_rng = np.random.default_rng(CONFIG['sampler']['seed'] + epoch)\\n\\n    selected_seen = np.zeros(node_count, dtype=bool)\\n    computation_seen = np.zeros(node_count, dtype=bool)\\n    edge_seen = np.zeros(train_rows, dtype=bool)\\n    layer_totals = [\\n        {'layer': layer + 1, 'requested_k_per_batch': int(CONFIG['sampler']['k_l'][layer]),\\n         'candidate_node_slots': 0, 'selected_context_node_slots': 0, 'directed_block_entries': 0}\\n        for layer in range(CONFIG['model']['layers'])\\n    ]\\n    selected_item_slots = Counter()\\n    sampler_seconds = 0.0\\n    propagation_seconds = 0.0\\n    examples = 0\\n    batches = 0\\n    ranking_loss_sum = 0.0\\n    regularization_sum = 0.0\\n\\n    model.train()\\n    for batch_number, start in enumerate(range(0, train_rows, CONFIG['training']['batch_size'])):\\n        stop = min(start + CONFIG['training']['batch_size'], train_rows)\\n        batch_indices = order[start:stop]\\n        users_np = train_users[batch_indices]\\n        positives_np = train_items[batch_indices]\\n        negatives_np = negatives_all[batch_indices]\\n        v0 = np.unique(np.concatenate((\\n            users_np.astype(np.int64),\\n            user_count + positives_np.astype(np.int64),\\n            user_count + negatives_np.astype(np.int64),\\n        )))\\n\\n        sampler_started = time.perf_counter()\\n        trace = build_trace(v0, sampler_rng)\\n        sampler_seconds += time.perf_counter() - sampler_started\\n        contract = trace_contract(trace)\\n        all_trace_contracts.append(contract)\\n        if not all(contract.values()):\\n            raise AssertionError(contract)\\n\\n        if epoch == 0 and batch_number == 0:\\n            replay_rng = np.random.default_rng(CONFIG['sampler']['seed'])\\n            replay = build_trace(v0, replay_rng)\\n            deterministic_replay_passed = trace_fingerprint(trace) == trace_fingerprint(replay)\\n            if not deterministic_replay_passed:\\n                raise AssertionError('First-batch deterministic sampler replay fail')\\n\\n        update_trace_accounting(\\n            trace, selected_seen, computation_seen, edge_seen, layer_totals, selected_item_slots\\n        )\\n        user_loc, positive_loc, negative_loc = endpoint_locations(v0, users_np, positives_np, negatives_np)\\n\\n        torch.cuda.synchronize(device)\\n        propagation_started = time.perf_counter()\\n        optimizer.zero_grad(set_to_none=True)\\n        propagated_v0 = model.sampled_propagate(trace)\\n        user_loc_t = torch.from_numpy(user_loc).to(device=device, dtype=torch.long)\\n        positive_loc_t = torch.from_numpy(positive_loc).to(device=device, dtype=torch.long)\\n        negative_loc_t = torch.from_numpy(negative_loc).to(device=device, dtype=torch.long)\\n        user_vec = propagated_v0[user_loc_t]\\n        positive_vec = propagated_v0[positive_loc_t]\\n        negative_vec = propagated_v0[negative_loc_t]\\n        ranking_loss = -F.logsigmoid(\\n            (user_vec * positive_vec).sum(dim=1) - (user_vec * negative_vec).sum(dim=1)\\n        ).mean()\\n\\n        endpoint_global = np.concatenate((\\n            users_np.astype(np.int64),\\n            user_count + positives_np.astype(np.int64),\\n            user_count + negatives_np.astype(np.int64),\\n        ))\\n        ego = model.embedding(torch.from_numpy(endpoint_global).to(device=device, dtype=torch.long))\\n        batch_size_actual = stop - start\\n        ego = ego.reshape(3, batch_size_actual, -1)\\n        regularization = ego.square().sum(dim=2).sum(dim=0).mean()\\n        loss = ranking_loss + CONFIG['training']['l2_coefficient'] * regularization\\n        if not torch.isfinite(loss):\\n            raise FloatingPointError(f'Loss không hữu hạn ở epoch {epoch + 1}, batch {batch_number + 1}')\\n        loss.backward()\\n        if model.embedding.weight.grad is None or not model.embedding.weight.grad.is_sparse:\\n            raise AssertionError('Sampled embedding gradient phải sparse')\\n        if not torch.isfinite(model.embedding.weight.grad._values()).all():\\n            raise FloatingPointError('Sparse embedding gradient không hữu hạn')\\n        optimizer.step()\\n        optimizer_steps += 1\\n        torch.cuda.synchronize(device)\\n        propagation_seconds += time.perf_counter() - propagation_started\\n\\n        ranking_loss_sum += float(ranking_loss.detach().cpu()) * batch_size_actual\\n        regularization_sum += float(regularization.detach().cpu()) * batch_size_actual\\n        examples += batch_size_actual\\n        batches += 1\\n        if batch_number % 10 == 0:\\n            print(f'Epoch {epoch + 1}: batch {batch_number + 1}, examples {examples:,}/{train_rows:,}')\\n\\n        del trace, propagated_v0, user_vec, positive_vec, negative_vec, ego, loss, ranking_loss, regularization\\n        if batch_number % 10 == 0:\\n            gc.collect()\\n\\n    if examples != train_rows:\\n        raise AssertionError('Epoch không nhìn đủ training edge')\\n    mean_loss = (\\n        ranking_loss_sum / train_rows\\n        + CONFIG['training']['l2_coefficient'] * regularization_sum / train_rows\\n    )\\n    record = {\\n        'epoch': epoch + 1,\\n        'input_fingerprint_seconds': fingerprint_seconds,\\n        'mean_loss': float(mean_loss),\\n        'examples': int(examples),\\n        'optimizer_steps': int(batches),\\n        'wall_seconds': time.perf_counter() - epoch_started,\\n        'sampler_seconds': sampler_seconds,\\n        'propagation_and_update_seconds': propagation_seconds,\\n        'throughput_examples_per_second': examples / (time.perf_counter() - epoch_started),\\n        'unique_selected_context_nodes': int(selected_seen.sum()),\\n        'unique_computation_nodes': int(computation_seen.sum()),\\n        'unique_training_edges_in_blocks': int(edge_seen.sum()),\\n        'layer_totals': layer_totals,\\n        'selected_item_context_slots_by_cohort': dict(selected_item_slots),\\n    }\\n    epoch_records.append(record)\\n    print(json.dumps(record, ensure_ascii=False, indent=2))\\n    del order, negatives_all, selected_seen, computation_seen, edge_seen\\n    gc.collect()\\n    torch.cuda.empty_cache()\\n\\ntraining_wall_seconds = time.perf_counter() - training_started\\ntraining_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)\\nparameters_changed = not torch.equal(initial_probe, model.embedding.weight[:1024].detach().cpu())\\nexpected_batches = math.ceil(train_rows / CONFIG['training']['batch_size'])\\ntrace_assertions = {\\n    key: all(contract[key] for contract in all_trace_contracts)\\n    for key in all_trace_contracts[0]\\n}\\ntraining_assertions = {\\n    'cuda_used': device.type == 'cuda',\\n    'all_registered_epochs_completed': len(epoch_records) == CONFIG['training']['epochs'],\\n    'each_epoch_saw_all_training_edges_as_positive_pairs': all(r['examples'] == train_rows for r in epoch_records),\\n    'optimizer_steps_match_batches': optimizer_steps == expected_batches * CONFIG['training']['epochs'],\\n    'losses_finite': all(np.isfinite(r['mean_loss']) for r in epoch_records),\\n    'embedding_parameters_changed': parameters_changed,\\n    'negative_sampler_policy_enforced': True,\\n    'fixed_last_epoch_used': True,\\n    'deterministic_first_batch_sampler_replay': deterministic_replay_passed,\\n    'sampled_training_used': True,\\n    'positive_edge_retain_policy_used': CONFIG['sampler']['positive_edge_policy'] == 'retain',\\n    'rectangular_sampled_local_normalization_used': CONFIG['sampler']['normalization'] == 'rectangular_sampled_local_bi_normalization',\\n    **trace_assertions,\\n}\\nif not all(training_assertions.values()):\\n    raise AssertionError(training_assertions)\\n\\noptimizer.zero_grad(set_to_none=True)\\ndel optimizer, all_trace_contracts\\ngc.collect()\\ntorch.cuda.empty_cache()\\n\",\"evaluation\":\"print('Dựng full normalized training adjacency cho validation inference...')\\nglobal_users = train_users.astype(np.int64)\\nglobal_items = user_count + train_items.astype(np.int64)\\nfull_weights = (1.0 / np.sqrt(user_degrees[train_users] * item_degrees[train_items])).astype(np.float32)\\nfull_src = np.concatenate((global_users, global_items))\\nfull_dst = np.concatenate((global_items, global_users))\\nfull_values = np.concatenate((full_weights, full_weights))\\nfull_indices = np.stack((full_dst, full_src), axis=0)\\nfull_adjacency = torch.sparse_coo_tensor(\\n    torch.from_numpy(full_indices).to(device=device, dtype=torch.long),\\n    torch.from_numpy(full_values).to(device=device),\\n    size=(node_count, node_count),\\n    device=device,\\n).coalesce()\\nif full_adjacency._nnz() != 2 * train_rows or not torch.isfinite(full_adjacency.values()).all():\\n    raise AssertionError('Full inference adjacency không hợp lệ')\\ndel global_users, global_items, full_weights, full_src, full_dst, full_values, full_indices\\ngc.collect()\\n\\n\\ndef resolve_boundary_ties(scores, top_values, top_indices, k):\\n    cutoff = top_values[:, -1]\\n    total_at_cutoff = (scores == cutoff[:, None]).sum(dim=1)\\n    selected_at_cutoff = (top_values == cutoff[:, None]).sum(dim=1)\\n    boundary_rows = torch.nonzero(total_at_cutoff != selected_at_cutoff, as_tuple=False).flatten()\\n    if boundary_rows.numel() == 0:\\n        return top_indices, 0\\n    resolved = top_indices.clone()\\n    for row in boundary_rows.tolist():\\n        row_scores = scores[row]\\n        row_cutoff = cutoff[row]\\n        better = torch.nonzero(row_scores > row_cutoff, as_tuple=False).flatten()\\n        tied = torch.nonzero(row_scores == row_cutoff, as_tuple=False).flatten()\\n        chosen = torch.cat((better, tied[:k - int(better.numel())]))\\n        if chosen.numel() != k:\\n            raise AssertionError('Không resolve được top-k boundary tie')\\n        resolved[row] = chosen\\n    return resolved, int(boundary_rows.numel())\\n\\n\\nmodel.eval()\\nk = CONFIG['evaluation']['k']\\neval_batch_size = CONFIG['evaluation']['eval_batch_size']\\nranks = np.empty(validation_rows, dtype=np.int32)\\nrecommended_mask = np.zeros(item_count, dtype=bool)\\nexposure_counts = Counter()\\nboundary_tie_rows = 0\\nitem_ids = torch.arange(item_count, device=device, dtype=torch.long)\\n\\ntorch.cuda.reset_peak_memory_stats(device)\\nevaluation_started = time.perf_counter()\\nwith torch.no_grad():\\n    final_embeddings = model.full_propagate(full_adjacency)\\n    item_matrix = final_embeddings[user_count:]\\n    for start in range(0, validation_rows, eval_batch_size):\\n        stop = min(start + eval_batch_size, validation_rows)\\n        users = torch.from_numpy(target_users[start:stop]).to(device=device, dtype=torch.long)\\n        targets = torch.from_numpy(target_items[start:stop]).to(device=device, dtype=torch.long)\\n        scores = final_embeddings[users] @ item_matrix.T\\n\\n        mask_rows = []\\n        mask_items = []\\n        for local_row, history in enumerate(histories[start:stop]):\\n            if history.size:\\n                mask_rows.append(np.full(history.size, local_row, dtype=np.int64))\\n                mask_items.append(history.astype(np.int64, copy=False))\\n        if mask_rows:\\n            scores[\\n                torch.from_numpy(np.concatenate(mask_rows)).to(device),\\n                torch.from_numpy(np.concatenate(mask_items)).to(device),\\n            ] = -torch.inf\\n\\n        row_ids = torch.arange(stop - start, device=device)\\n        target_scores = scores[row_ids, targets]\\n        if not torch.isfinite(target_scores).all():\\n            raise AssertionError('Target score bị mask hoặc không hữu hạn')\\n        strictly_better = (scores > target_scores[:, None]).sum(dim=1)\\n        equal_and_lower_id = ((scores == target_scores[:, None]) & (item_ids[None, :] < targets[:, None])).sum(dim=1)\\n        ranks[start:stop] = (1 + strictly_better + equal_and_lower_id).cpu().numpy().astype(np.int32)\\n\\n        top_values, top_indices = torch.topk(scores, k=k, dim=1, largest=True, sorted=True)\\n        top_indices, resolved_rows = resolve_boundary_ties(scores, top_values, top_indices, k)\\n        boundary_tie_rows += resolved_rows\\n        top_numpy = top_indices.cpu().numpy()\\n        recommended_mask[top_numpy.reshape(-1)] = True\\n        top_degrees = item_degrees[top_numpy.reshape(-1)]\\n        exposure_counts['head'] += int((top_degrees >= 397).sum())\\n        exposure_counts['body'] += int(((top_degrees >= 13) & (top_degrees <= 396)).sum())\\n        exposure_counts['tail'] += int((top_degrees <= 12).sum())\\n        if (start // eval_batch_size) % 100 == 0:\\n            print(f'Evaluated {stop:,}/{validation_rows:,} target')\\n\\ntorch.cuda.synchronize(device)\\nevaluation_wall_seconds = time.perf_counter() - evaluation_started\\nevaluation_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)\\noverall_metrics = row_metrics(ranks, k)\\nitem_labels = np.asarray([item_cohort(item_degrees[item]) for item in target_items])\\nuser_labels = np.asarray([user_cohort(user_degrees[user]) for user in target_users])\\nitem_cohort_metrics = grouped_metrics(ranks, item_labels, k)\\nuser_cohort_metrics = grouped_metrics(ranks, user_labels, k)\\nrecommendation_slots = validation_rows * k\\ncoverage = float(recommended_mask.mean())\\n\\nmostpop_metrics = mostpop['metrics']\\nbpr_metrics = bpr['validation']['metrics']\\nfull_metrics = full_lightgcn['validation']['metrics']\\nevaluation_assertions = {\\n    'full_graph_inference_used': True,\\n    'full_inference_adjacency_has_two_directions_per_edge': full_adjacency._nnz() == 2 * train_rows,\\n    'every_validation_target_ranked': ranks.size == validation_rows,\\n    'all_ranks_within_candidate_count': bool(np.all(ranks >= 1) and np.all(ranks <= target_candidate_counts)),\\n    'recommendation_slots_reconcile': sum(exposure_counts.values()) == recommendation_slots,\\n    'catalog_coverage_reconciles': abs(coverage - int(recommended_mask.sum()) / item_count) < 1e-15,\\n    'same_validation_rows_as_all_references': validation_rows == int(mostpop_metrics['rows']) == int(bpr_metrics['rows']) == int(full_metrics['rows']),\\n    'rank_tie_break_applied': True,\\n    'topk_boundary_ties_resolved': True,\\n    'test_targets_not_read_during_evaluation': True,\\n}\\nif not all(evaluation_assertions.values()):\\n    raise AssertionError(evaluation_assertions)\\n\\nsummary = {\\n    'status': 'UNIFORM_SAMPLING_VALIDATION_SMOKE_EXECUTED',\\n    'registered_config': CONFIG,\\n    'source': {\\n        'manifest_path': str(MANIFEST_PATH),\\n        'train_edges': {'path': str(train_path), 'rows': train_rows, 'sha256': train_sha},\\n        'validation_targets': {'path': str(validation_path), 'rows': validation_rows, 'sha256': validation_sha},\\n        'mostpop_summary': {'path': str(MOSTPOP_PATH), 'sha256': MOSTPOP_SUMMARY_SHA256},\\n        'bpr_mf_summary': {'path': str(BPR_PATH), 'sha256': BPR_SUMMARY_SHA256},\\n        'full_lightgcn_summary': {'path': str(FULL_LIGHTGCN_PATH), 'sha256': FULL_LIGHTGCN_SUMMARY_SHA256},\\n        'users': user_count, 'items': item_count, 'nodes': node_count,\\n        'directed_training_csr_entries': int(graph_csr.nnz),\\n    },\\n    'integrity_assertions': {**source_assertions, **training_assertions, **evaluation_assertions},\\n    'training': {\\n        'epochs': epoch_records,\\n        'wall_seconds': training_wall_seconds,\\n        'peak_gpu_memory_mb': training_peak_gpu_mb,\\n        'negative_draws': train_rows * CONFIG['training']['epochs'],\\n        'negative_resamples': total_negative_resamples,\\n        'optimizer_steps': optimizer_steps,\\n        'deterministic_first_batch_trace_passed': deterministic_replay_passed,\\n    },\\n    'validation': {\\n        'metrics': overall_metrics,\\n        'target_item_cohorts': item_cohort_metrics,\\n        'user_activity_cohorts': user_cohort_metrics,\\n        'recommendation_exposure': {\\n            'recommendation_slots': recommendation_slots,\\n            'unique_items_at_k': int(recommended_mask.sum()),\\n            'catalog_coverage_at_k': coverage,\\n            'item_cohort_share': {label: exposure_counts[label] / recommendation_slots for label in ('head', 'body', 'tail')},\\n        },\\n        'wall_seconds': evaluation_wall_seconds,\\n        'peak_gpu_memory_mb': evaluation_peak_gpu_mb,\\n        'topk_boundary_tie_rows': boundary_tie_rows,\\n    },\\n    'reference_comparison': {\\n        'comparison_boundary': 'MostPop, BPR-MF and full LightGCN are context references. Only the future M1 degree-aware run with matched_control_id static-controls-k65536-smoke-v1 is budget matched to this M0 run.',\\n        'mostpop': {'ndcg_at_20': mostpop_metrics['ndcg_at_k'], 'recall_at_20': mostpop_metrics['recall_at_k']},\\n        'bpr_mf': {'ndcg_at_20': bpr_metrics['ndcg_at_k'], 'recall_at_20': bpr_metrics['recall_at_k']},\\n        'full_lightgcn': {'ndcg_at_20': full_metrics['ndcg_at_k'], 'recall_at_20': full_metrics['recall_at_k']},\\n        'uniform_m0': {'ndcg_at_20': overall_metrics['ndcg_at_k'], 'recall_at_20': overall_metrics['recall_at_k']},\\n    },\\n    'environment': {\\n        'python': platform.python_version(), 'platform': platform.platform(),\\n        'torch': torch.__version__, 'cuda': torch.version.cuda,\\n        'gpu': torch.cuda.get_device_name(device), 'process_peak_rss_mb': process_peak_rss_mb(),\\n    },\\n    'claim_boundary': CONFIG['claim_boundary'],\\n}\\nprint(json.dumps({\\n    'status': summary['status'],\\n    'metrics': overall_metrics,\\n    'coverage_at_20': coverage,\\n    'training_wall_seconds': training_wall_seconds,\\n    'training_peak_gpu_mb': training_peak_gpu_mb,\\n    'all_assertions_pass': all(summary['integrity_assertions'].values()),\\n}, ensure_ascii=False, indent=2))\\n\"},\"M1\":{\"training\":\"import torch\\nfrom torch import nn\\nimport torch.nn.functional as F\\n\\nif not torch.cuda.is_available():\\n    raise RuntimeError('Degree-aware sampling smoke yêu cầu Colab GPU CUDA.')\\ndevice = torch.device('cuda')\\ntorch.backends.cuda.matmul.allow_tf32 = False\\n\\n\\n@dataclass\\nclass NumpyBlock:\\n    source_nodes: np.ndarray\\n    target_nodes: np.ndarray\\n    source_local: np.ndarray\\n    target_local: np.ndarray\\n    weights: np.ndarray\\n    edge_ids: np.ndarray\\n\\n\\ndef candidate_nodes(previous_nodes):\\n    rows = graph_csr[previous_nodes]\\n    neighbors = np.unique(rows.indices.astype(np.int64, copy=False))\\n    positions = np.searchsorted(previous_nodes, neighbors)\\n    in_previous = np.zeros(neighbors.size, dtype=bool)\\n    bounded = positions < previous_nodes.size\\n    in_previous[bounded] = previous_nodes[positions[bounded]] == neighbors[bounded]\\n    return neighbors[~in_previous]\\n\\n\\ndef degree_exact_k(candidates, requested_k, rng, degrees, exponent):\\n    effective_k = min(int(requested_k), int(candidates.size))\\n    if effective_k == 0:\\n        return np.empty(0, dtype=np.int64)\\n    raw_degrees = np.asarray(degrees[candidates], dtype=np.float64)\\n    weights = np.power(raw_degrees, float(exponent))\\n    if not np.isfinite(weights).all() or np.any(weights <= 0):\\n        raise ValueError('Degree proposal yêu cầu mọi candidate weight hữu hạn và dương')\\n    uniforms = np.clip(rng.random(candidates.size), np.finfo(np.float64).tiny, 1.0 - np.finfo(np.float64).eps)\\n    gumbels = -np.log(-np.log(uniforms))\\n    priorities = np.log(weights) + gumbels\\n    if effective_k == candidates.size:\\n        selected = candidates.copy()\\n    else:\\n        threshold = candidates.size - effective_k\\n        selected = candidates[np.argpartition(priorities, threshold)[threshold:]]\\n    selected.sort()\\n    return selected\\n\\n\\ndef build_block(source_nodes, target_nodes):\\n    sliced = graph_csr[target_nodes]\\n    source_global = sliced.indices.astype(np.int64, copy=False)\\n    target_local_all = np.repeat(np.arange(target_nodes.size, dtype=np.int64), np.diff(sliced.indptr))\\n    positions = np.searchsorted(source_nodes, source_global)\\n    keep = positions < source_nodes.size\\n    valid_positions = positions[keep]\\n    valid_globals = source_global[keep]\\n    matched = source_nodes[valid_positions] == valid_globals\\n    keep_indices = np.flatnonzero(keep)[matched]\\n    source_local = positions[keep_indices].astype(np.int64, copy=False)\\n    target_local = target_local_all[keep_indices].astype(np.int64, copy=False)\\n    underlying_edge_ids = sliced.data[keep_indices].astype(np.int64, copy=False) - 1\\n    if source_local.size == 0:\\n        weights = np.empty(0, dtype=np.float32)\\n    else:\\n        source_degree = np.bincount(source_local, minlength=source_nodes.size)\\n        target_degree = np.bincount(target_local, minlength=target_nodes.size)\\n        weights = (1.0 / np.sqrt(source_degree[source_local] * target_degree[target_local])).astype(np.float32)\\n    return NumpyBlock(source_nodes, target_nodes, source_local, target_local, weights, underlying_edge_ids)\\n\\n\\ndef build_trace(v0, rng):\\n    k_sets = [v0]\\n    blocks = []\\n    layers = []\\n    previous = v0\\n    for layer_index, requested_k in enumerate(CONFIG['sampler']['k_l'], start=1):\\n        candidates = candidate_nodes(previous)\\n        selected = degree_exact_k(\\n            candidates, requested_k, rng, node_degrees, CONFIG['sampler']['degree_exponent']\\n        )\\n        current = np.union1d(v0, selected)\\n        block = build_block(current, previous)\\n        layers.append({\\n            'layer': layer_index,\\n            'candidate_nodes': candidates,\\n            'selected_nodes': selected,\\n            'requested_k': int(requested_k),\\n        })\\n        k_sets.append(current)\\n        blocks.append(block)\\n        previous = current\\n    return {'v0': v0, 'k_sets': k_sets, 'blocks': blocks, 'layers': layers}\\n\\n\\ndef trace_fingerprint(trace):\\n    digest = hashlib.sha256()\\n    for layer, block in zip(trace['layers'], trace['blocks']):\\n        for array in (layer['candidate_nodes'], layer['selected_nodes'], block.edge_ids):\\n            digest.update(np.asarray(array, dtype=np.int64).tobytes())\\n    return digest.hexdigest()\\n\\n\\ndef trace_contract(trace):\\n    exact_k = True\\n    unique = True\\n    disjoint = True\\n    state_rule = True\\n    original_edges = True\\n    no_self_loop = True\\n    finite_weights = True\\n    proposal_weights_positive = True\\n    for index, (layer, block) in enumerate(zip(trace['layers'], trace['blocks']), start=1):\\n        candidates = layer['candidate_nodes']\\n        selected = layer['selected_nodes']\\n        previous = trace['k_sets'][index - 1]\\n        expected_current = np.union1d(trace['v0'], selected)\\n        exact_k &= selected.size == min(layer['requested_k'], candidates.size)\\n        unique &= np.unique(selected).size == selected.size\\n        disjoint &= np.intersect1d(selected, previous).size == 0\\n        state_rule &= np.array_equal(trace['k_sets'][index], expected_current)\\n        original_edges &= bool(np.all((block.edge_ids >= 0) & (block.edge_ids < train_rows)))\\n        if block.source_local.size:\\n            no_self_loop &= not np.any(block.source_nodes[block.source_local] == block.target_nodes[block.target_local])\\n        finite_weights &= bool(np.isfinite(block.weights).all())\\n        candidate_weights = np.power(\\n            node_degrees[candidates].astype(np.float64), CONFIG['sampler']['degree_exponent']\\n        )\\n        proposal_weights_positive &= bool(np.isfinite(candidate_weights).all() and np.all(candidate_weights > 0))\\n    return {\\n        'exact_k_every_layer': bool(exact_k),\\n        'sampled_nodes_unique': bool(unique),\\n        'sampled_nodes_disjoint_from_previous_K': bool(disjoint),\\n        'K_l_equals_V0_union_V_l': bool(state_rule),\\n        'sampled_blocks_use_only_registered_training_edges': bool(original_edges),\\n        'sampled_blocks_have_no_self_loop': bool(no_self_loop),\\n        'sampled_block_weights_finite': bool(finite_weights),\\n        'degree_proposal_weights_positive': bool(proposal_weights_positive),\\n    }\\n\\n\\ndef block_to_torch(block):\\n    indices = torch.from_numpy(np.stack((block.target_local, block.source_local))).to(device=device, dtype=torch.long)\\n    values = torch.from_numpy(block.weights).to(device=device)\\n    return torch.sparse_coo_tensor(\\n        indices, values,\\n        size=(block.target_nodes.size, block.source_nodes.size),\\n        device=device,\\n        is_coalesced=False,\\n    ).coalesce()\\n\\n\\ndef verify_rectangular_direction_oracle():\\n    indices = torch.tensor([[0, 1], [0, 1]], device=device)\\n    values = torch.tensor([1.0, 1.0], device=device)\\n    block = torch.sparse_coo_tensor(indices, values, (2, 2), device=device).coalesce()\\n    source = torch.tensor([[3.0], [7.0]], device=device)\\n    torch.testing.assert_close(torch.sparse.mm(block, source), source)\\n\\n\\nverify_rectangular_direction_oracle()\\n\\n\\nclass SampledLightGCN(nn.Module):\\n    def __init__(self, nodes, embedding_dim, layers, init_std):\\n        super().__init__()\\n        self.embedding = nn.Embedding(nodes, embedding_dim, sparse=True)\\n        self.layers = int(layers)\\n        nn.init.normal_(self.embedding.weight, std=init_std)\\n\\n    def sampled_propagate(self, trace):\\n        torch_blocks = [block_to_torch(block) for block in trace['blocks']]\\n        depth_outputs = [self.embedding(torch.from_numpy(trace['v0']).to(device=device, dtype=torch.long))]\\n        for depth in range(1, self.layers + 1):\\n            current_nodes = trace['k_sets'][depth]\\n            hidden = self.embedding(torch.from_numpy(current_nodes).to(device=device, dtype=torch.long))\\n            for layer_index in range(depth - 1, -1, -1):\\n                hidden = torch.sparse.mm(torch_blocks[layer_index], hidden)\\n            depth_outputs.append(hidden)\\n        return torch.stack(depth_outputs, dim=0).mean(dim=0)\\n\\n    def full_propagate(self, normalized_adjacency):\\n        current = self.embedding.weight\\n        combined = current / (self.layers + 1)\\n        for _ in range(self.layers):\\n            current = torch.sparse.mm(normalized_adjacency, current)\\n            combined = combined + current / (self.layers + 1)\\n        return combined\\n\\n\\ndef endpoint_locations(v0, users, positives, negatives):\\n    global_users = users.astype(np.int64, copy=False)\\n    global_positives = user_count + positives.astype(np.int64, copy=False)\\n    global_negatives = user_count + negatives.astype(np.int64, copy=False)\\n    locations = [np.searchsorted(v0, values) for values in (global_users, global_positives, global_negatives)]\\n    for values, found in zip((global_users, global_positives, global_negatives), locations):\\n        if not np.array_equal(v0[found], values):\\n            raise AssertionError('Triplet endpoint không nằm trong V0')\\n    return locations\\n\\n\\ndef update_trace_accounting(trace, selected_seen, computation_seen, edge_seen, layer_totals, selected_item_slots):\\n    computation_seen[trace['v0']] = True\\n    for index, (layer, block) in enumerate(zip(trace['layers'], trace['blocks'])):\\n        selected = layer['selected_nodes']\\n        selected_seen[selected] = True\\n        computation_seen[trace['k_sets'][index + 1]] = True\\n        edge_seen[block.edge_ids] = True\\n        layer_totals[index]['candidate_node_slots'] += int(layer['candidate_nodes'].size)\\n        layer_totals[index]['selected_context_node_slots'] += int(selected.size)\\n        layer_totals[index]['directed_block_entries'] += int(block.edge_ids.size)\\n        selected_items = selected[selected >= user_count] - user_count\\n        if selected_items.size:\\n            degrees = item_degrees[selected_items]\\n            selected_item_slots['head'] += int((degrees >= 397).sum())\\n            selected_item_slots['body'] += int(((degrees >= 13) & (degrees <= 396)).sum())\\n            selected_item_slots['tail'] += int((degrees <= 12).sum())\\n\\n\\ntorch.manual_seed(CONFIG['training']['seed'])\\ntorch.cuda.manual_seed_all(CONFIG['training']['seed'])\\nmodel = SampledLightGCN(\\n    node_count,\\n    CONFIG['model']['embedding_dim'],\\n    CONFIG['model']['layers'],\\n    CONFIG['model']['initialization_std'],\\n).to(device)\\ninitial_probe = model.embedding.weight[:1024].detach().cpu().clone()\\ninitial_embedding_sha256 = embedding_sha256(model.embedding.weight)\\nepoch_input_fingerprints = []\\noptimizer = torch.optim.SparseAdam(model.parameters(), lr=CONFIG['training']['learning_rate'])\\n\\ntorch.cuda.reset_peak_memory_stats(device)\\ntraining_started = time.perf_counter()\\nepoch_records = []\\noptimizer_steps = 0\\ntotal_negative_resamples = 0\\ndeterministic_replay_passed = False\\nall_trace_contracts = []\\n\\nfor epoch in range(CONFIG['training']['epochs']):\\n    epoch_started = time.perf_counter()\\n    pair_rng = np.random.default_rng(CONFIG['training']['seed'] + epoch)\\n    order = pair_rng.permutation(train_rows)\\n    negatives_all, resampled = sample_exact_uniform_negatives(train_users, pair_rng, positive_keys, item_count)\\n    total_negative_resamples += resampled\\n    fingerprint_started = time.perf_counter()\\n    epoch_input_fingerprints.append({\\n        'epoch': epoch + 1,\\n        'pair_order_sha256': array_sha256(order),\\n        'negative_items_sha256': array_sha256(negatives_all),\\n    })\\n    fingerprint_seconds = time.perf_counter() - fingerprint_started\\n    sampler_rng = np.random.default_rng(CONFIG['sampler']['seed'] + epoch)\\n\\n    selected_seen = np.zeros(node_count, dtype=bool)\\n    computation_seen = np.zeros(node_count, dtype=bool)\\n    edge_seen = np.zeros(train_rows, dtype=bool)\\n    layer_totals = [\\n        {'layer': layer + 1, 'requested_k_per_batch': int(CONFIG['sampler']['k_l'][layer]),\\n         'candidate_node_slots': 0, 'selected_context_node_slots': 0, 'directed_block_entries': 0}\\n        for layer in range(CONFIG['model']['layers'])\\n    ]\\n    selected_item_slots = Counter()\\n    sampler_seconds = 0.0\\n    propagation_seconds = 0.0\\n    examples = 0\\n    batches = 0\\n    ranking_loss_sum = 0.0\\n    regularization_sum = 0.0\\n\\n    model.train()\\n    for batch_number, start in enumerate(range(0, train_rows, CONFIG['training']['batch_size'])):\\n        stop = min(start + CONFIG['training']['batch_size'], train_rows)\\n        batch_indices = order[start:stop]\\n        users_np = train_users[batch_indices]\\n        positives_np = train_items[batch_indices]\\n        negatives_np = negatives_all[batch_indices]\\n        v0 = np.unique(np.concatenate((\\n            users_np.astype(np.int64),\\n            user_count + positives_np.astype(np.int64),\\n            user_count + negatives_np.astype(np.int64),\\n        )))\\n\\n        sampler_started = time.perf_counter()\\n        trace = build_trace(v0, sampler_rng)\\n        sampler_seconds += time.perf_counter() - sampler_started\\n        contract = trace_contract(trace)\\n        all_trace_contracts.append(contract)\\n        if not all(contract.values()):\\n            raise AssertionError(contract)\\n\\n        if epoch == 0 and batch_number == 0:\\n            replay_rng = np.random.default_rng(CONFIG['sampler']['seed'])\\n            replay = build_trace(v0, replay_rng)\\n            deterministic_replay_passed = trace_fingerprint(trace) == trace_fingerprint(replay)\\n            if not deterministic_replay_passed:\\n                raise AssertionError('First-batch deterministic sampler replay fail')\\n\\n        update_trace_accounting(\\n            trace, selected_seen, computation_seen, edge_seen, layer_totals, selected_item_slots\\n        )\\n        user_loc, positive_loc, negative_loc = endpoint_locations(v0, users_np, positives_np, negatives_np)\\n\\n        torch.cuda.synchronize(device)\\n        propagation_started = time.perf_counter()\\n        optimizer.zero_grad(set_to_none=True)\\n        propagated_v0 = model.sampled_propagate(trace)\\n        user_loc_t = torch.from_numpy(user_loc).to(device=device, dtype=torch.long)\\n        positive_loc_t = torch.from_numpy(positive_loc).to(device=device, dtype=torch.long)\\n        negative_loc_t = torch.from_numpy(negative_loc).to(device=device, dtype=torch.long)\\n        user_vec = propagated_v0[user_loc_t]\\n        positive_vec = propagated_v0[positive_loc_t]\\n        negative_vec = propagated_v0[negative_loc_t]\\n        ranking_loss = -F.logsigmoid(\\n            (user_vec * positive_vec).sum(dim=1) - (user_vec * negative_vec).sum(dim=1)\\n        ).mean()\\n\\n        endpoint_global = np.concatenate((\\n            users_np.astype(np.int64),\\n            user_count + positives_np.astype(np.int64),\\n            user_count + negatives_np.astype(np.int64),\\n        ))\\n        ego = model.embedding(torch.from_numpy(endpoint_global).to(device=device, dtype=torch.long))\\n        batch_size_actual = stop - start\\n        ego = ego.reshape(3, batch_size_actual, -1)\\n        regularization = ego.square().sum(dim=2).sum(dim=0).mean()\\n        loss = ranking_loss + CONFIG['training']['l2_coefficient'] * regularization\\n        if not torch.isfinite(loss):\\n            raise FloatingPointError(f'Loss không hữu hạn ở epoch {epoch + 1}, batch {batch_number + 1}')\\n        loss.backward()\\n        if model.embedding.weight.grad is None or not model.embedding.weight.grad.is_sparse:\\n            raise AssertionError('Sampled embedding gradient phải sparse')\\n        if not torch.isfinite(model.embedding.weight.grad._values()).all():\\n            raise FloatingPointError('Sparse embedding gradient không hữu hạn')\\n        optimizer.step()\\n        optimizer_steps += 1\\n        torch.cuda.synchronize(device)\\n        propagation_seconds += time.perf_counter() - propagation_started\\n\\n        ranking_loss_sum += float(ranking_loss.detach().cpu()) * batch_size_actual\\n        regularization_sum += float(regularization.detach().cpu()) * batch_size_actual\\n        examples += batch_size_actual\\n        batches += 1\\n        if batch_number % 10 == 0:\\n            print(f'Epoch {epoch + 1}: batch {batch_number + 1}, examples {examples:,}/{train_rows:,}')\\n\\n        del trace, propagated_v0, user_vec, positive_vec, negative_vec, ego, loss, ranking_loss, regularization\\n        if batch_number % 10 == 0:\\n            gc.collect()\\n\\n    if examples != train_rows:\\n        raise AssertionError('Epoch không nhìn đủ training edge')\\n    mean_loss = (\\n        ranking_loss_sum / train_rows\\n        + CONFIG['training']['l2_coefficient'] * regularization_sum / train_rows\\n    )\\n    record = {\\n        'epoch': epoch + 1,\\n        'input_fingerprint_seconds': fingerprint_seconds,\\n        'mean_loss': float(mean_loss),\\n        'examples': int(examples),\\n        'optimizer_steps': int(batches),\\n        'wall_seconds': time.perf_counter() - epoch_started,\\n        'sampler_seconds': sampler_seconds,\\n        'propagation_and_update_seconds': propagation_seconds,\\n        'throughput_examples_per_second': examples / (time.perf_counter() - epoch_started),\\n        'unique_selected_context_nodes': int(selected_seen.sum()),\\n        'unique_computation_nodes': int(computation_seen.sum()),\\n        'unique_training_edges_in_blocks': int(edge_seen.sum()),\\n        'layer_totals': layer_totals,\\n        'selected_item_context_slots_by_cohort': dict(selected_item_slots),\\n    }\\n    epoch_records.append(record)\\n    print(json.dumps(record, ensure_ascii=False, indent=2))\\n    del order, negatives_all, selected_seen, computation_seen, edge_seen\\n    gc.collect()\\n    torch.cuda.empty_cache()\\n\\ntraining_wall_seconds = time.perf_counter() - training_started\\ntraining_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)\\nparameters_changed = not torch.equal(initial_probe, model.embedding.weight[:1024].detach().cpu())\\nexpected_batches = math.ceil(train_rows / CONFIG['training']['batch_size'])\\ntrace_assertions = {\\n    key: all(contract[key] for contract in all_trace_contracts)\\n    for key in all_trace_contracts[0]\\n}\\ntraining_assertions = {\\n    'cuda_used': device.type == 'cuda',\\n    'all_registered_epochs_completed': len(epoch_records) == CONFIG['training']['epochs'],\\n    'each_epoch_saw_all_training_edges_as_positive_pairs': all(r['examples'] == train_rows for r in epoch_records),\\n    'optimizer_steps_match_batches': optimizer_steps == expected_batches * CONFIG['training']['epochs'],\\n    'losses_finite': all(np.isfinite(r['mean_loss']) for r in epoch_records),\\n    'embedding_parameters_changed': parameters_changed,\\n    'negative_sampler_policy_enforced': True,\\n    'fixed_last_epoch_used': True,\\n    'deterministic_first_batch_sampler_replay': deterministic_replay_passed,\\n    'sampled_training_used': True,\\n    'positive_edge_retain_policy_used': CONFIG['sampler']['positive_edge_policy'] == 'retain',\\n    'rectangular_sampled_local_normalization_used': CONFIG['sampler']['normalization'] == 'rectangular_sampled_local_bi_normalization',\\n    **trace_assertions,\\n}\\nif not all(training_assertions.values()):\\n    raise AssertionError(training_assertions)\\n\\noptimizer.zero_grad(set_to_none=True)\\ndel optimizer, all_trace_contracts\\ngc.collect()\\ntorch.cuda.empty_cache()\\n\",\"evaluation\":\"print('Dựng full normalized training adjacency cho validation inference...')\\nglobal_users = train_users.astype(np.int64)\\nglobal_items = user_count + train_items.astype(np.int64)\\nfull_weights = (1.0 / np.sqrt(user_degrees[train_users] * item_degrees[train_items])).astype(np.float32)\\nfull_src = np.concatenate((global_users, global_items))\\nfull_dst = np.concatenate((global_items, global_users))\\nfull_values = np.concatenate((full_weights, full_weights))\\nfull_indices = np.stack((full_dst, full_src), axis=0)\\nfull_adjacency = torch.sparse_coo_tensor(\\n    torch.from_numpy(full_indices).to(device=device, dtype=torch.long),\\n    torch.from_numpy(full_values).to(device=device),\\n    size=(node_count, node_count),\\n    device=device,\\n).coalesce()\\nif full_adjacency._nnz() != 2 * train_rows or not torch.isfinite(full_adjacency.values()).all():\\n    raise AssertionError('Full inference adjacency không hợp lệ')\\ndel global_users, global_items, full_weights, full_src, full_dst, full_values, full_indices\\ngc.collect()\\n\\n\\ndef resolve_boundary_ties(scores, top_values, top_indices, k):\\n    cutoff = top_values[:, -1]\\n    total_at_cutoff = (scores == cutoff[:, None]).sum(dim=1)\\n    selected_at_cutoff = (top_values == cutoff[:, None]).sum(dim=1)\\n    boundary_rows = torch.nonzero(total_at_cutoff != selected_at_cutoff, as_tuple=False).flatten()\\n    if boundary_rows.numel() == 0:\\n        return top_indices, 0\\n    resolved = top_indices.clone()\\n    for row in boundary_rows.tolist():\\n        row_scores = scores[row]\\n        row_cutoff = cutoff[row]\\n        better = torch.nonzero(row_scores > row_cutoff, as_tuple=False).flatten()\\n        tied = torch.nonzero(row_scores == row_cutoff, as_tuple=False).flatten()\\n        chosen = torch.cat((better, tied[:k - int(better.numel())]))\\n        if chosen.numel() != k:\\n            raise AssertionError('Không resolve được top-k boundary tie')\\n        resolved[row] = chosen\\n    return resolved, int(boundary_rows.numel())\\n\\n\\nmodel.eval()\\nk = CONFIG['evaluation']['k']\\neval_batch_size = CONFIG['evaluation']['eval_batch_size']\\nranks = np.empty(validation_rows, dtype=np.int32)\\nrecommended_mask = np.zeros(item_count, dtype=bool)\\nexposure_counts = Counter()\\nboundary_tie_rows = 0\\nitem_ids = torch.arange(item_count, device=device, dtype=torch.long)\\n\\ntorch.cuda.reset_peak_memory_stats(device)\\nevaluation_started = time.perf_counter()\\nwith torch.no_grad():\\n    final_embeddings = model.full_propagate(full_adjacency)\\n    item_matrix = final_embeddings[user_count:]\\n    for start in range(0, validation_rows, eval_batch_size):\\n        stop = min(start + eval_batch_size, validation_rows)\\n        users = torch.from_numpy(target_users[start:stop]).to(device=device, dtype=torch.long)\\n        targets = torch.from_numpy(target_items[start:stop]).to(device=device, dtype=torch.long)\\n        scores = final_embeddings[users] @ item_matrix.T\\n\\n        mask_rows = []\\n        mask_items = []\\n        for local_row, history in enumerate(histories[start:stop]):\\n            if history.size:\\n                mask_rows.append(np.full(history.size, local_row, dtype=np.int64))\\n                mask_items.append(history.astype(np.int64, copy=False))\\n        if mask_rows:\\n            scores[\\n                torch.from_numpy(np.concatenate(mask_rows)).to(device),\\n                torch.from_numpy(np.concatenate(mask_items)).to(device),\\n            ] = -torch.inf\\n\\n        row_ids = torch.arange(stop - start, device=device)\\n        target_scores = scores[row_ids, targets]\\n        if not torch.isfinite(target_scores).all():\\n            raise AssertionError('Target score bị mask hoặc không hữu hạn')\\n        strictly_better = (scores > target_scores[:, None]).sum(dim=1)\\n        equal_and_lower_id = ((scores == target_scores[:, None]) & (item_ids[None, :] < targets[:, None])).sum(dim=1)\\n        ranks[start:stop] = (1 + strictly_better + equal_and_lower_id).cpu().numpy().astype(np.int32)\\n\\n        top_values, top_indices = torch.topk(scores, k=k, dim=1, largest=True, sorted=True)\\n        top_indices, resolved_rows = resolve_boundary_ties(scores, top_values, top_indices, k)\\n        boundary_tie_rows += resolved_rows\\n        top_numpy = top_indices.cpu().numpy()\\n        recommended_mask[top_numpy.reshape(-1)] = True\\n        top_degrees = item_degrees[top_numpy.reshape(-1)]\\n        exposure_counts['head'] += int((top_degrees >= 397).sum())\\n        exposure_counts['body'] += int(((top_degrees >= 13) & (top_degrees <= 396)).sum())\\n        exposure_counts['tail'] += int((top_degrees <= 12).sum())\\n        if (start // eval_batch_size) % 100 == 0:\\n            print(f'Evaluated {stop:,}/{validation_rows:,} target')\\n\\ntorch.cuda.synchronize(device)\\nevaluation_wall_seconds = time.perf_counter() - evaluation_started\\nevaluation_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)\\noverall_metrics = row_metrics(ranks, k)\\nitem_labels = np.asarray([item_cohort(item_degrees[item]) for item in target_items])\\nuser_labels = np.asarray([user_cohort(user_degrees[user]) for user in target_users])\\nitem_cohort_metrics = grouped_metrics(ranks, item_labels, k)\\nuser_cohort_metrics = grouped_metrics(ranks, user_labels, k)\\nrecommendation_slots = validation_rows * k\\ncoverage = float(recommended_mask.mean())\\n\\nmostpop_metrics = mostpop['metrics']\\nbpr_metrics = bpr['validation']['metrics']\\nfull_metrics = full_lightgcn['validation']['metrics']\\nuniform_metrics = uniform_m0['validation']['metrics']\\nevaluation_assertions = {\\n    'full_graph_inference_used': True,\\n    'full_inference_adjacency_has_two_directions_per_edge': full_adjacency._nnz() == 2 * train_rows,\\n    'every_validation_target_ranked': ranks.size == validation_rows,\\n    'all_ranks_within_candidate_count': bool(np.all(ranks >= 1) and np.all(ranks <= target_candidate_counts)),\\n    'recommendation_slots_reconcile': sum(exposure_counts.values()) == recommendation_slots,\\n    'catalog_coverage_reconciles': abs(coverage - int(recommended_mask.sum()) / item_count) < 1e-15,\\n    'same_validation_rows_as_all_references': validation_rows == int(mostpop_metrics['rows']) == int(bpr_metrics['rows']) == int(full_metrics['rows']) == int(uniform_metrics['rows']),\\n    'same_gpu_as_uniform_control': torch.cuda.get_device_name(device) == uniform_m0['environment']['gpu'],\\n    'rank_tie_break_applied': True,\\n    'topk_boundary_ties_resolved': True,\\n    'test_targets_not_read_during_evaluation': True,\\n}\\nif not all(evaluation_assertions.values()):\\n    raise AssertionError(evaluation_assertions)\\n\\nsummary = {\\n    'status': 'DEGREE_AWARE_SAMPLING_VALIDATION_SMOKE_EXECUTED',\\n    'registered_config': CONFIG,\\n    'source': {\\n        'manifest_path': str(MANIFEST_PATH),\\n        'train_edges': {'path': str(train_path), 'rows': train_rows, 'sha256': train_sha},\\n        'validation_targets': {'path': str(validation_path), 'rows': validation_rows, 'sha256': validation_sha},\\n        'mostpop_summary': {'path': str(MOSTPOP_PATH), 'sha256': MOSTPOP_SUMMARY_SHA256},\\n        'bpr_mf_summary': {'path': str(BPR_PATH), 'sha256': BPR_SUMMARY_SHA256},\\n        'full_lightgcn_summary': {'path': str(FULL_LIGHTGCN_PATH), 'sha256': FULL_LIGHTGCN_SUMMARY_SHA256},\\n        'uniform_m0_summary': {'path': str(UNIFORM_PATH), 'sha256': UNIFORM_SUMMARY_SHA256},\\n        'users': user_count, 'items': item_count, 'nodes': node_count,\\n        'directed_training_csr_entries': int(graph_csr.nnz),\\n    },\\n    'integrity_assertions': {**source_assertions, **training_assertions, **evaluation_assertions},\\n    'training': {\\n        'epochs': epoch_records,\\n        'wall_seconds': training_wall_seconds,\\n        'peak_gpu_memory_mb': training_peak_gpu_mb,\\n        'negative_draws': train_rows * CONFIG['training']['epochs'],\\n        'negative_resamples': total_negative_resamples,\\n        'optimizer_steps': optimizer_steps,\\n        'deterministic_first_batch_trace_passed': deterministic_replay_passed,\\n    },\\n    'validation': {\\n        'metrics': overall_metrics,\\n        'target_item_cohorts': item_cohort_metrics,\\n        'user_activity_cohorts': user_cohort_metrics,\\n        'recommendation_exposure': {\\n            'recommendation_slots': recommendation_slots,\\n            'unique_items_at_k': int(recommended_mask.sum()),\\n            'catalog_coverage_at_k': coverage,\\n            'item_cohort_share': {label: exposure_counts[label] / recommendation_slots for label in ('head', 'body', 'tail')},\\n        },\\n        'wall_seconds': evaluation_wall_seconds,\\n        'peak_gpu_memory_mb': evaluation_peak_gpu_mb,\\n        'topk_boundary_tie_rows': boundary_tie_rows,\\n    },\\n    'matched_comparison': {\\n        'comparison_boundary': 'M0 uniform is the only budget-matched static control. Deltas are descriptive for one fixed validation seed; no significance or test-set claim.',\\n        'uniform_m0': {\\n            'ndcg_at_20': uniform_metrics['ndcg_at_k'],\\n            'recall_at_20': uniform_metrics['recall_at_k'],\\n            'hits_at_20': uniform_metrics['hits_at_k'],\\n            'catalog_coverage_at_20': uniform_m0['validation']['recommendation_exposure']['catalog_coverage_at_k'],\\n            'training_wall_seconds': uniform_m0['training']['wall_seconds'],\\n            'training_peak_gpu_memory_mb': uniform_m0['training']['peak_gpu_memory_mb'],\\n        },\\n        'degree_m1': {\\n            'ndcg_at_20': overall_metrics['ndcg_at_k'],\\n            'recall_at_20': overall_metrics['recall_at_k'],\\n            'hits_at_20': overall_metrics['hits_at_k'],\\n            'catalog_coverage_at_20': coverage,\\n            'training_wall_seconds': training_wall_seconds,\\n            'training_peak_gpu_memory_mb': training_peak_gpu_mb,\\n        },\\n        'degree_minus_uniform': {\\n            'ndcg_at_20': overall_metrics['ndcg_at_k'] - uniform_metrics['ndcg_at_k'],\\n            'recall_at_20': overall_metrics['recall_at_k'] - uniform_metrics['recall_at_k'],\\n            'hits_at_20': overall_metrics['hits_at_k'] - uniform_metrics['hits_at_k'],\\n            'catalog_coverage_at_20': coverage - uniform_m0['validation']['recommendation_exposure']['catalog_coverage_at_k'],\\n            'training_wall_seconds': training_wall_seconds - uniform_m0['training']['wall_seconds'],\\n            'training_peak_gpu_memory_mb': training_peak_gpu_mb - uniform_m0['training']['peak_gpu_memory_mb'],\\n        },\\n    },\\n    'reference_comparison': {\\n        'comparison_boundary': 'MostPop, BPR-MF and full LightGCN are context references, not budget-matched controls.',\\n        'mostpop': {'ndcg_at_20': mostpop_metrics['ndcg_at_k'], 'recall_at_20': mostpop_metrics['recall_at_k']},\\n        'bpr_mf': {'ndcg_at_20': bpr_metrics['ndcg_at_k'], 'recall_at_20': bpr_metrics['recall_at_k']},\\n        'full_lightgcn': {'ndcg_at_20': full_metrics['ndcg_at_k'], 'recall_at_20': full_metrics['recall_at_k']},\\n        'uniform_m0': {'ndcg_at_20': uniform_metrics['ndcg_at_k'], 'recall_at_20': uniform_metrics['recall_at_k']},\\n        'degree_m1': {'ndcg_at_20': overall_metrics['ndcg_at_k'], 'recall_at_20': overall_metrics['recall_at_k']},\\n    },\\n    'environment': {\\n        'python': platform.python_version(), 'platform': platform.platform(),\\n        'torch': torch.__version__, 'cuda': torch.version.cuda,\\n        'gpu': torch.cuda.get_device_name(device), 'process_peak_rss_mb': process_peak_rss_mb(),\\n    },\\n    'claim_boundary': CONFIG['claim_boundary'],\\n}\\nprint(json.dumps({\\n    'status': summary['status'],\\n    'metrics': overall_metrics,\\n    'coverage_at_20': coverage,\\n    'training_wall_seconds': training_wall_seconds,\\n    'training_peak_gpu_mb': training_peak_gpu_mb,\\n    'all_assertions_pass': all(summary['integrity_assertions'].values()),\\n}, ensure_ascii=False, indent=2))\\n\"},\"M2\":{\"training\":\"import torch\\nfrom torch import nn\\nimport torch.nn.functional as F\\n\\nif not torch.cuda.is_available():\\n    raise RuntimeError('Frontier-normalized sampling smoke yêu cầu Colab GPU CUDA.')\\ndevice = torch.device('cuda')\\ntorch.backends.cuda.matmul.allow_tf32 = False\\n\\n\\n@dataclass\\nclass NumpyBlock:\\n    source_nodes: np.ndarray\\n    target_nodes: np.ndarray\\n    source_local: np.ndarray\\n    target_local: np.ndarray\\n    weights: np.ndarray\\n    edge_ids: np.ndarray\\n\\n\\ndef candidate_nodes_with_support(previous_nodes):\\n    rows = graph_csr[previous_nodes]\\n    neighbors, support = np.unique(\\n        rows.indices.astype(np.int64, copy=False), return_counts=True\\n    )\\n    positions = np.searchsorted(previous_nodes, neighbors)\\n    in_previous = np.zeros(neighbors.size, dtype=bool)\\n    bounded = positions < previous_nodes.size\\n    in_previous[bounded] = previous_nodes[positions[bounded]] == neighbors[bounded]\\n    keep = ~in_previous\\n    return neighbors[keep], support[keep].astype(np.int64, copy=False)\\n\\n\\ndef frontier_normalized_weights(candidates, support, degrees, support_exponent, degree_exponent):\\n    candidates = np.asarray(candidates, dtype=np.int64)\\n    support = np.asarray(support, dtype=np.float64)\\n    if candidates.size != support.size:\\n        raise ValueError('Candidate và frontier support phải cùng kích thước')\\n    raw_degrees = np.asarray(degrees[candidates], dtype=np.float64)\\n    weights = np.power(support, float(support_exponent)) * np.power(raw_degrees, float(degree_exponent))\\n    if not np.isfinite(weights).all() or np.any(weights <= 0):\\n        raise ValueError('Frontier-normalized proposal yêu cầu support và degree tạo weight hữu hạn, dương')\\n    return weights\\n\\n\\ndef frontier_normalized_exact_k(candidates, support, requested_k, rng, degrees, support_exponent, degree_exponent):\\n    effective_k = min(int(requested_k), int(candidates.size))\\n    if effective_k == 0:\\n        return np.empty(0, dtype=np.int64)\\n    weights = frontier_normalized_weights(\\n        candidates, support, degrees, support_exponent, degree_exponent\\n    )\\n    uniforms = np.clip(rng.random(candidates.size), np.finfo(np.float64).tiny, 1.0 - np.finfo(np.float64).eps)\\n    gumbels = -np.log(-np.log(uniforms))\\n    priorities = np.log(weights) + gumbels\\n    if effective_k == candidates.size:\\n        selected = candidates.copy()\\n    else:\\n        threshold = candidates.size - effective_k\\n        selected = candidates[np.argpartition(priorities, threshold)[threshold:]]\\n    selected.sort()\\n    return selected\\n\\n\\ndef build_block(source_nodes, target_nodes):\\n    sliced = graph_csr[target_nodes]\\n    source_global = sliced.indices.astype(np.int64, copy=False)\\n    target_local_all = np.repeat(np.arange(target_nodes.size, dtype=np.int64), np.diff(sliced.indptr))\\n    positions = np.searchsorted(source_nodes, source_global)\\n    keep = positions < source_nodes.size\\n    valid_positions = positions[keep]\\n    valid_globals = source_global[keep]\\n    matched = source_nodes[valid_positions] == valid_globals\\n    keep_indices = np.flatnonzero(keep)[matched]\\n    source_local = positions[keep_indices].astype(np.int64, copy=False)\\n    target_local = target_local_all[keep_indices].astype(np.int64, copy=False)\\n    underlying_edge_ids = sliced.data[keep_indices].astype(np.int64, copy=False) - 1\\n    if source_local.size == 0:\\n        weights = np.empty(0, dtype=np.float32)\\n    else:\\n        source_degree = np.bincount(source_local, minlength=source_nodes.size)\\n        target_degree = np.bincount(target_local, minlength=target_nodes.size)\\n        weights = (1.0 / np.sqrt(source_degree[source_local] * target_degree[target_local])).astype(np.float32)\\n    return NumpyBlock(source_nodes, target_nodes, source_local, target_local, weights, underlying_edge_ids)\\n\\n\\ndef build_trace(v0, rng):\\n    k_sets = [v0]\\n    blocks = []\\n    layers = []\\n    previous = v0\\n    for layer_index, requested_k in enumerate(CONFIG['sampler']['k_l'], start=1):\\n        candidates, candidate_support = candidate_nodes_with_support(previous)\\n        candidate_weights = frontier_normalized_weights(\\n            candidates, candidate_support, node_degrees,\\n            CONFIG['sampler']['support_exponent'], CONFIG['sampler']['degree_exponent'],\\n        )\\n        selected = frontier_normalized_exact_k(\\n            candidates, candidate_support, requested_k, rng, node_degrees,\\n            CONFIG['sampler']['support_exponent'], CONFIG['sampler']['degree_exponent'],\\n        )\\n        selected_positions = np.searchsorted(candidates, selected)\\n        current = np.union1d(v0, selected)\\n        block = build_block(current, previous)\\n        layers.append({\\n            'layer': layer_index,\\n            'candidate_nodes': candidates,\\n            'candidate_support': candidate_support,\\n            'candidate_weights': candidate_weights,\\n            'selected_nodes': selected,\\n            'selected_support': candidate_support[selected_positions],\\n            'requested_k': int(requested_k),\\n        })\\n        k_sets.append(current)\\n        blocks.append(block)\\n        previous = current\\n    return {'v0': v0, 'k_sets': k_sets, 'blocks': blocks, 'layers': layers}\\n\\n\\ndef trace_fingerprint(trace):\\n    digest = hashlib.sha256()\\n    for layer, block in zip(trace['layers'], trace['blocks']):\\n        for array in (layer['candidate_nodes'], layer['candidate_support'], layer['selected_nodes'], block.edge_ids):\\n            digest.update(np.asarray(array, dtype=np.int64).tobytes())\\n    return digest.hexdigest()\\n\\n\\ndef trace_contract(trace):\\n    exact_k = True\\n    unique = True\\n    disjoint = True\\n    state_rule = True\\n    original_edges = True\\n    no_self_loop = True\\n    finite_weights = True\\n    proposal_weights_positive = True\\n    frontier_support_positive = True\\n    for index, (layer, block) in enumerate(zip(trace['layers'], trace['blocks']), start=1):\\n        candidates = layer['candidate_nodes']\\n        selected = layer['selected_nodes']\\n        previous = trace['k_sets'][index - 1]\\n        expected_current = np.union1d(trace['v0'], selected)\\n        exact_k &= selected.size == min(layer['requested_k'], candidates.size)\\n        unique &= np.unique(selected).size == selected.size\\n        disjoint &= np.intersect1d(selected, previous).size == 0\\n        state_rule &= np.array_equal(trace['k_sets'][index], expected_current)\\n        original_edges &= bool(np.all((block.edge_ids >= 0) & (block.edge_ids < train_rows)))\\n        if block.source_local.size:\\n            no_self_loop &= not np.any(block.source_nodes[block.source_local] == block.target_nodes[block.target_local])\\n        finite_weights &= bool(np.isfinite(block.weights).all())\\n        candidate_weights = frontier_normalized_weights(\\n            candidates, layer['candidate_support'], node_degrees,\\n            CONFIG['sampler']['support_exponent'], CONFIG['sampler']['degree_exponent'],\\n        )\\n        proposal_weights_positive &= bool(np.isfinite(candidate_weights).all() and np.all(candidate_weights > 0))\\n        frontier_support_positive &= bool(\\n            layer['candidate_support'].size == candidates.size\\n            and np.all(layer['candidate_support'] > 0)\\n        )\\n    return {\\n        'exact_k_every_layer': bool(exact_k),\\n        'sampled_nodes_unique': bool(unique),\\n        'sampled_nodes_disjoint_from_previous_K': bool(disjoint),\\n        'K_l_equals_V0_union_V_l': bool(state_rule),\\n        'sampled_blocks_use_only_registered_training_edges': bool(original_edges),\\n        'sampled_blocks_have_no_self_loop': bool(no_self_loop),\\n        'sampled_block_weights_finite': bool(finite_weights),\\n        'frontier_proposal_weights_positive': bool(proposal_weights_positive),\\n        'frontier_support_positive': bool(frontier_support_positive),\\n    }\\n\\n\\ndef block_to_torch(block):\\n    indices = torch.from_numpy(np.stack((block.target_local, block.source_local))).to(device=device, dtype=torch.long)\\n    values = torch.from_numpy(block.weights).to(device=device)\\n    return torch.sparse_coo_tensor(\\n        indices, values,\\n        size=(block.target_nodes.size, block.source_nodes.size),\\n        device=device,\\n        is_coalesced=False,\\n    ).coalesce()\\n\\n\\ndef verify_rectangular_direction_oracle():\\n    indices = torch.tensor([[0, 1], [0, 1]], device=device)\\n    values = torch.tensor([1.0, 1.0], device=device)\\n    block = torch.sparse_coo_tensor(indices, values, (2, 2), device=device).coalesce()\\n    source = torch.tensor([[3.0], [7.0]], device=device)\\n    torch.testing.assert_close(torch.sparse.mm(block, source), source)\\n\\n\\nverify_rectangular_direction_oracle()\\n\\n\\nclass SampledLightGCN(nn.Module):\\n    def __init__(self, nodes, embedding_dim, layers, init_std):\\n        super().__init__()\\n        self.embedding = nn.Embedding(nodes, embedding_dim, sparse=True)\\n        self.layers = int(layers)\\n        nn.init.normal_(self.embedding.weight, std=init_std)\\n\\n    def sampled_propagate(self, trace):\\n        torch_blocks = [block_to_torch(block) for block in trace['blocks']]\\n        depth_outputs = [self.embedding(torch.from_numpy(trace['v0']).to(device=device, dtype=torch.long))]\\n        for depth in range(1, self.layers + 1):\\n            current_nodes = trace['k_sets'][depth]\\n            hidden = self.embedding(torch.from_numpy(current_nodes).to(device=device, dtype=torch.long))\\n            for layer_index in range(depth - 1, -1, -1):\\n                hidden = torch.sparse.mm(torch_blocks[layer_index], hidden)\\n            depth_outputs.append(hidden)\\n        return torch.stack(depth_outputs, dim=0).mean(dim=0)\\n\\n    def full_propagate(self, normalized_adjacency):\\n        current = self.embedding.weight\\n        combined = current / (self.layers + 1)\\n        for _ in range(self.layers):\\n            current = torch.sparse.mm(normalized_adjacency, current)\\n            combined = combined + current / (self.layers + 1)\\n        return combined\\n\\n\\ndef endpoint_locations(v0, users, positives, negatives):\\n    global_users = users.astype(np.int64, copy=False)\\n    global_positives = user_count + positives.astype(np.int64, copy=False)\\n    global_negatives = user_count + negatives.astype(np.int64, copy=False)\\n    locations = [np.searchsorted(v0, values) for values in (global_users, global_positives, global_negatives)]\\n    for values, found in zip((global_users, global_positives, global_negatives), locations):\\n        if not np.array_equal(v0[found], values):\\n            raise AssertionError('Triplet endpoint không nằm trong V0')\\n    return locations\\n\\n\\ndef update_trace_accounting(trace, selected_seen, computation_seen, edge_seen, layer_totals, selected_item_slots):\\n    computation_seen[trace['v0']] = True\\n    for index, (layer, block) in enumerate(zip(trace['layers'], trace['blocks'])):\\n        selected = layer['selected_nodes']\\n        selected_seen[selected] = True\\n        computation_seen[trace['k_sets'][index + 1]] = True\\n        edge_seen[block.edge_ids] = True\\n        layer_totals[index]['candidate_node_slots'] += int(layer['candidate_nodes'].size)\\n        layer_totals[index]['selected_context_node_slots'] += int(selected.size)\\n        layer_totals[index]['directed_block_entries'] += int(block.edge_ids.size)\\n        selected_positions = np.searchsorted(layer['candidate_nodes'], selected)\\n        layer_totals[index]['candidate_support_mass'] += int(layer['candidate_support'].sum())\\n        layer_totals[index]['selected_support_mass'] += int(layer['selected_support'].sum())\\n        layer_totals[index]['candidate_degree_mass'] += int(node_degrees[layer['candidate_nodes']].sum())\\n        layer_totals[index]['selected_degree_mass'] += int(node_degrees[selected].sum())\\n        layer_totals[index]['candidate_weight_mass'] += float(layer['candidate_weights'].sum())\\n        layer_totals[index]['selected_weight_mass'] += float(layer['candidate_weights'][selected_positions].sum())\\n        selected_items = selected[selected >= user_count] - user_count\\n        if selected_items.size:\\n            degrees = item_degrees[selected_items]\\n            selected_item_slots['head'] += int((degrees >= 397).sum())\\n            selected_item_slots['body'] += int(((degrees >= 13) & (degrees <= 396)).sum())\\n            selected_item_slots['tail'] += int((degrees <= 12).sum())\\n\\n\\ntorch.manual_seed(CONFIG['training']['seed'])\\ntorch.cuda.manual_seed_all(CONFIG['training']['seed'])\\nmodel = SampledLightGCN(\\n    node_count,\\n    CONFIG['model']['embedding_dim'],\\n    CONFIG['model']['layers'],\\n    CONFIG['model']['initialization_std'],\\n).to(device)\\ninitial_probe = model.embedding.weight[:1024].detach().cpu().clone()\\ninitial_embedding_sha256 = embedding_sha256(model.embedding.weight)\\nepoch_input_fingerprints = []\\noptimizer = torch.optim.SparseAdam(model.parameters(), lr=CONFIG['training']['learning_rate'])\\n\\ntorch.cuda.reset_peak_memory_stats(device)\\ntraining_started = time.perf_counter()\\nepoch_records = []\\noptimizer_steps = 0\\ntotal_negative_resamples = 0\\ndeterministic_replay_passed = False\\nall_trace_contracts = []\\n\\nfor epoch in range(CONFIG['training']['epochs']):\\n    epoch_started = time.perf_counter()\\n    pair_rng = np.random.default_rng(CONFIG['training']['seed'] + epoch)\\n    order = pair_rng.permutation(train_rows)\\n    negatives_all, resampled = sample_exact_uniform_negatives(train_users, pair_rng, positive_keys, item_count)\\n    total_negative_resamples += resampled\\n    fingerprint_started = time.perf_counter()\\n    epoch_input_fingerprints.append({\\n        'epoch': epoch + 1,\\n        'pair_order_sha256': array_sha256(order),\\n        'negative_items_sha256': array_sha256(negatives_all),\\n    })\\n    fingerprint_seconds = time.perf_counter() - fingerprint_started\\n    sampler_rng = np.random.default_rng(CONFIG['sampler']['seed'] + epoch)\\n\\n    selected_seen = np.zeros(node_count, dtype=bool)\\n    computation_seen = np.zeros(node_count, dtype=bool)\\n    edge_seen = np.zeros(train_rows, dtype=bool)\\n    layer_totals = [\\n        {'layer': layer + 1, 'requested_k_per_batch': int(CONFIG['sampler']['k_l'][layer]),\\n         'candidate_node_slots': 0, 'selected_context_node_slots': 0, 'directed_block_entries': 0,\\n         'candidate_support_mass': 0, 'selected_support_mass': 0,\\n         'candidate_degree_mass': 0, 'selected_degree_mass': 0,\\n         'candidate_weight_mass': 0.0, 'selected_weight_mass': 0.0}\\n        for layer in range(CONFIG['model']['layers'])\\n    ]\\n    selected_item_slots = Counter()\\n    sampler_seconds = 0.0\\n    propagation_seconds = 0.0\\n    examples = 0\\n    batches = 0\\n    ranking_loss_sum = 0.0\\n    regularization_sum = 0.0\\n\\n    model.train()\\n    for batch_number, start in enumerate(range(0, train_rows, CONFIG['training']['batch_size'])):\\n        stop = min(start + CONFIG['training']['batch_size'], train_rows)\\n        batch_indices = order[start:stop]\\n        users_np = train_users[batch_indices]\\n        positives_np = train_items[batch_indices]\\n        negatives_np = negatives_all[batch_indices]\\n        v0 = np.unique(np.concatenate((\\n            users_np.astype(np.int64),\\n            user_count + positives_np.astype(np.int64),\\n            user_count + negatives_np.astype(np.int64),\\n        )))\\n\\n        sampler_started = time.perf_counter()\\n        trace = build_trace(v0, sampler_rng)\\n        sampler_seconds += time.perf_counter() - sampler_started\\n        contract = trace_contract(trace)\\n        all_trace_contracts.append(contract)\\n        if not all(contract.values()):\\n            raise AssertionError(contract)\\n\\n        if epoch == 0 and batch_number == 0:\\n            replay_rng = np.random.default_rng(CONFIG['sampler']['seed'])\\n            replay = build_trace(v0, replay_rng)\\n            deterministic_replay_passed = trace_fingerprint(trace) == trace_fingerprint(replay)\\n            if not deterministic_replay_passed:\\n                raise AssertionError('First-batch deterministic sampler replay fail')\\n\\n        update_trace_accounting(\\n            trace, selected_seen, computation_seen, edge_seen, layer_totals, selected_item_slots\\n        )\\n        user_loc, positive_loc, negative_loc = endpoint_locations(v0, users_np, positives_np, negatives_np)\\n\\n        torch.cuda.synchronize(device)\\n        propagation_started = time.perf_counter()\\n        optimizer.zero_grad(set_to_none=True)\\n        propagated_v0 = model.sampled_propagate(trace)\\n        user_loc_t = torch.from_numpy(user_loc).to(device=device, dtype=torch.long)\\n        positive_loc_t = torch.from_numpy(positive_loc).to(device=device, dtype=torch.long)\\n        negative_loc_t = torch.from_numpy(negative_loc).to(device=device, dtype=torch.long)\\n        user_vec = propagated_v0[user_loc_t]\\n        positive_vec = propagated_v0[positive_loc_t]\\n        negative_vec = propagated_v0[negative_loc_t]\\n        ranking_loss = -F.logsigmoid(\\n            (user_vec * positive_vec).sum(dim=1) - (user_vec * negative_vec).sum(dim=1)\\n        ).mean()\\n\\n        endpoint_global = np.concatenate((\\n            users_np.astype(np.int64),\\n            user_count + positives_np.astype(np.int64),\\n            user_count + negatives_np.astype(np.int64),\\n        ))\\n        ego = model.embedding(torch.from_numpy(endpoint_global).to(device=device, dtype=torch.long))\\n        batch_size_actual = stop - start\\n        ego = ego.reshape(3, batch_size_actual, -1)\\n        regularization = ego.square().sum(dim=2).sum(dim=0).mean()\\n        loss = ranking_loss + CONFIG['training']['l2_coefficient'] * regularization\\n        if not torch.isfinite(loss):\\n            raise FloatingPointError(f'Loss không hữu hạn ở epoch {epoch + 1}, batch {batch_number + 1}')\\n        loss.backward()\\n        if model.embedding.weight.grad is None or not model.embedding.weight.grad.is_sparse:\\n            raise AssertionError('Sampled embedding gradient phải sparse')\\n        if not torch.isfinite(model.embedding.weight.grad._values()).all():\\n            raise FloatingPointError('Sparse embedding gradient không hữu hạn')\\n        optimizer.step()\\n        optimizer_steps += 1\\n        torch.cuda.synchronize(device)\\n        propagation_seconds += time.perf_counter() - propagation_started\\n\\n        ranking_loss_sum += float(ranking_loss.detach().cpu()) * batch_size_actual\\n        regularization_sum += float(regularization.detach().cpu()) * batch_size_actual\\n        examples += batch_size_actual\\n        batches += 1\\n        if batch_number % 10 == 0:\\n            print(f'Epoch {epoch + 1}: batch {batch_number + 1}, examples {examples:,}/{train_rows:,}')\\n\\n        del trace, propagated_v0, user_vec, positive_vec, negative_vec, ego, loss, ranking_loss, regularization\\n        if batch_number % 10 == 0:\\n            gc.collect()\\n\\n    if examples != train_rows:\\n        raise AssertionError('Epoch không nhìn đủ training edge')\\n    mean_loss = (\\n        ranking_loss_sum / train_rows\\n        + CONFIG['training']['l2_coefficient'] * regularization_sum / train_rows\\n    )\\n    record = {\\n        'epoch': epoch + 1,\\n        'input_fingerprint_seconds': fingerprint_seconds,\\n        'mean_loss': float(mean_loss),\\n        'examples': int(examples),\\n        'optimizer_steps': int(batches),\\n        'wall_seconds': time.perf_counter() - epoch_started,\\n        'sampler_seconds': sampler_seconds,\\n        'propagation_and_update_seconds': propagation_seconds,\\n        'throughput_examples_per_second': examples / (time.perf_counter() - epoch_started),\\n        'unique_selected_context_nodes': int(selected_seen.sum()),\\n        'unique_computation_nodes': int(computation_seen.sum()),\\n        'unique_training_edges_in_blocks': int(edge_seen.sum()),\\n        'layer_totals': layer_totals,\\n        'selected_item_context_slots_by_cohort': dict(selected_item_slots),\\n    }\\n    epoch_records.append(record)\\n    print(json.dumps(record, ensure_ascii=False, indent=2))\\n    del order, negatives_all, selected_seen, computation_seen, edge_seen\\n    gc.collect()\\n    torch.cuda.empty_cache()\\n\\ntraining_wall_seconds = time.perf_counter() - training_started\\ntraining_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)\\nparameters_changed = not torch.equal(initial_probe, model.embedding.weight[:1024].detach().cpu())\\nexpected_batches = math.ceil(train_rows / CONFIG['training']['batch_size'])\\ntrace_assertions = {\\n    key: all(contract[key] for contract in all_trace_contracts)\\n    for key in all_trace_contracts[0]\\n}\\ntraining_assertions = {\\n    'cuda_used': device.type == 'cuda',\\n    'all_registered_epochs_completed': len(epoch_records) == CONFIG['training']['epochs'],\\n    'each_epoch_saw_all_training_edges_as_positive_pairs': all(r['examples'] == train_rows for r in epoch_records),\\n    'optimizer_steps_match_batches': optimizer_steps == expected_batches * CONFIG['training']['epochs'],\\n    'losses_finite': all(np.isfinite(r['mean_loss']) for r in epoch_records),\\n    'embedding_parameters_changed': parameters_changed,\\n    'negative_sampler_policy_enforced': True,\\n    'fixed_last_epoch_used': True,\\n    'deterministic_first_batch_sampler_replay': deterministic_replay_passed,\\n    'sampled_training_used': True,\\n    'positive_edge_retain_policy_used': CONFIG['sampler']['positive_edge_policy'] == 'retain',\\n    'rectangular_sampled_local_normalization_used': CONFIG['sampler']['normalization'] == 'rectangular_sampled_local_bi_normalization',\\n    **trace_assertions,\\n}\\nif not all(training_assertions.values()):\\n    raise AssertionError(training_assertions)\\n\\noptimizer.zero_grad(set_to_none=True)\\ndel optimizer, all_trace_contracts\\ngc.collect()\\ntorch.cuda.empty_cache()\\n\",\"evaluation\":\"print('Dựng full normalized training adjacency cho validation inference...')\\nglobal_users = train_users.astype(np.int64)\\nglobal_items = user_count + train_items.astype(np.int64)\\nfull_weights = (1.0 / np.sqrt(user_degrees[train_users] * item_degrees[train_items])).astype(np.float32)\\nfull_src = np.concatenate((global_users, global_items))\\nfull_dst = np.concatenate((global_items, global_users))\\nfull_values = np.concatenate((full_weights, full_weights))\\nfull_indices = np.stack((full_dst, full_src), axis=0)\\nfull_adjacency = torch.sparse_coo_tensor(\\n    torch.from_numpy(full_indices).to(device=device, dtype=torch.long),\\n    torch.from_numpy(full_values).to(device=device),\\n    size=(node_count, node_count),\\n    device=device,\\n).coalesce()\\nif full_adjacency._nnz() != 2 * train_rows or not torch.isfinite(full_adjacency.values()).all():\\n    raise AssertionError('Full inference adjacency không hợp lệ')\\ndel global_users, global_items, full_weights, full_src, full_dst, full_values, full_indices\\ngc.collect()\\n\\n\\ndef resolve_boundary_ties(scores, top_values, top_indices, k):\\n    cutoff = top_values[:, -1]\\n    total_at_cutoff = (scores == cutoff[:, None]).sum(dim=1)\\n    selected_at_cutoff = (top_values == cutoff[:, None]).sum(dim=1)\\n    boundary_rows = torch.nonzero(total_at_cutoff != selected_at_cutoff, as_tuple=False).flatten()\\n    if boundary_rows.numel() == 0:\\n        return top_indices, 0\\n    resolved = top_indices.clone()\\n    for row in boundary_rows.tolist():\\n        row_scores = scores[row]\\n        row_cutoff = cutoff[row]\\n        better = torch.nonzero(row_scores > row_cutoff, as_tuple=False).flatten()\\n        tied = torch.nonzero(row_scores == row_cutoff, as_tuple=False).flatten()\\n        chosen = torch.cat((better, tied[:k - int(better.numel())]))\\n        if chosen.numel() != k:\\n            raise AssertionError('Không resolve được top-k boundary tie')\\n        resolved[row] = chosen\\n    return resolved, int(boundary_rows.numel())\\n\\n\\nmodel.eval()\\nk = CONFIG['evaluation']['k']\\neval_batch_size = CONFIG['evaluation']['eval_batch_size']\\nranks = np.empty(validation_rows, dtype=np.int32)\\nrecommended_mask = np.zeros(item_count, dtype=bool)\\nexposure_counts = Counter()\\nboundary_tie_rows = 0\\nitem_ids = torch.arange(item_count, device=device, dtype=torch.long)\\n\\ntorch.cuda.reset_peak_memory_stats(device)\\nevaluation_started = time.perf_counter()\\nwith torch.no_grad():\\n    final_embeddings = model.full_propagate(full_adjacency)\\n    item_matrix = final_embeddings[user_count:]\\n    for start in range(0, validation_rows, eval_batch_size):\\n        stop = min(start + eval_batch_size, validation_rows)\\n        users = torch.from_numpy(target_users[start:stop]).to(device=device, dtype=torch.long)\\n        targets = torch.from_numpy(target_items[start:stop]).to(device=device, dtype=torch.long)\\n        scores = final_embeddings[users] @ item_matrix.T\\n\\n        mask_rows = []\\n        mask_items = []\\n        for local_row, history in enumerate(histories[start:stop]):\\n            if history.size:\\n                mask_rows.append(np.full(history.size, local_row, dtype=np.int64))\\n                mask_items.append(history.astype(np.int64, copy=False))\\n        if mask_rows:\\n            scores[\\n                torch.from_numpy(np.concatenate(mask_rows)).to(device),\\n                torch.from_numpy(np.concatenate(mask_items)).to(device),\\n            ] = -torch.inf\\n\\n        row_ids = torch.arange(stop - start, device=device)\\n        target_scores = scores[row_ids, targets]\\n        if not torch.isfinite(target_scores).all():\\n            raise AssertionError('Target score bị mask hoặc không hữu hạn')\\n        strictly_better = (scores > target_scores[:, None]).sum(dim=1)\\n        equal_and_lower_id = ((scores == target_scores[:, None]) & (item_ids[None, :] < targets[:, None])).sum(dim=1)\\n        ranks[start:stop] = (1 + strictly_better + equal_and_lower_id).cpu().numpy().astype(np.int32)\\n\\n        top_values, top_indices = torch.topk(scores, k=k, dim=1, largest=True, sorted=True)\\n        top_indices, resolved_rows = resolve_boundary_ties(scores, top_values, top_indices, k)\\n        boundary_tie_rows += resolved_rows\\n        top_numpy = top_indices.cpu().numpy()\\n        recommended_mask[top_numpy.reshape(-1)] = True\\n        top_degrees = item_degrees[top_numpy.reshape(-1)]\\n        exposure_counts['head'] += int((top_degrees >= 397).sum())\\n        exposure_counts['body'] += int(((top_degrees >= 13) & (top_degrees <= 396)).sum())\\n        exposure_counts['tail'] += int((top_degrees <= 12).sum())\\n        if (start // eval_batch_size) % 100 == 0:\\n            print(f'Evaluated {stop:,}/{validation_rows:,} target')\\n\\ntorch.cuda.synchronize(device)\\nevaluation_wall_seconds = time.perf_counter() - evaluation_started\\nevaluation_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)\\noverall_metrics = row_metrics(ranks, k)\\nitem_labels = np.asarray([item_cohort(item_degrees[item]) for item in target_items])\\nuser_labels = np.asarray([user_cohort(user_degrees[user]) for user in target_users])\\nitem_cohort_metrics = grouped_metrics(ranks, item_labels, k)\\nuser_cohort_metrics = grouped_metrics(ranks, user_labels, k)\\nrecommendation_slots = validation_rows * k\\ncoverage = float(recommended_mask.mean())\\n\\nmostpop_metrics = mostpop['metrics']\\nbpr_metrics = bpr['validation']['metrics']\\nfull_metrics = full_lightgcn['validation']['metrics']\\nuniform_metrics = uniform_m0['validation']['metrics']\\ndegree_metrics = degree_m1['validation']['metrics']\\nevaluation_assertions = {\\n    'full_graph_inference_used': True,\\n    'full_inference_adjacency_has_two_directions_per_edge': full_adjacency._nnz() == 2 * train_rows,\\n    'every_validation_target_ranked': ranks.size == validation_rows,\\n    'all_ranks_within_candidate_count': bool(np.all(ranks >= 1) and np.all(ranks <= target_candidate_counts)),\\n    'recommendation_slots_reconcile': sum(exposure_counts.values()) == recommendation_slots,\\n    'catalog_coverage_reconciles': abs(coverage - int(recommended_mask.sum()) / item_count) < 1e-15,\\n    'same_validation_rows_as_all_references': validation_rows == int(mostpop_metrics['rows']) == int(bpr_metrics['rows']) == int(full_metrics['rows']) == int(uniform_metrics['rows']) == int(degree_metrics['rows']),\\n    'same_gpu_as_uniform_control': torch.cuda.get_device_name(device) == uniform_m0['environment']['gpu'],\\n    'same_gpu_as_degree_control': torch.cuda.get_device_name(device) == degree_m1['environment']['gpu'],\\n    'rank_tie_break_applied': True,\\n    'topk_boundary_ties_resolved': True,\\n    'test_targets_not_read_during_evaluation': True,\\n}\\nif not all(evaluation_assertions.values()):\\n    raise AssertionError(evaluation_assertions)\\n\\nsummary = {\\n    'status': 'FRONTIER_NORMALIZED_SAMPLING_VALIDATION_SMOKE_EXECUTED',\\n    'registered_config': CONFIG,\\n    'source': {\\n        'manifest_path': str(MANIFEST_PATH),\\n        'train_edges': {'path': str(train_path), 'rows': train_rows, 'sha256': train_sha},\\n        'validation_targets': {'path': str(validation_path), 'rows': validation_rows, 'sha256': validation_sha},\\n        'mostpop_summary': {'path': str(MOSTPOP_PATH), 'sha256': MOSTPOP_SUMMARY_SHA256},\\n        'bpr_mf_summary': {'path': str(BPR_PATH), 'sha256': BPR_SUMMARY_SHA256},\\n        'full_lightgcn_summary': {'path': str(FULL_LIGHTGCN_PATH), 'sha256': FULL_LIGHTGCN_SUMMARY_SHA256},\\n        'uniform_m0_summary': {'path': str(UNIFORM_PATH), 'sha256': UNIFORM_SUMMARY_SHA256},\\n        'degree_m1_summary': {'path': str(DEGREE_PATH), 'sha256': DEGREE_SUMMARY_SHA256},\\n        'users': user_count, 'items': item_count, 'nodes': node_count,\\n        'directed_training_csr_entries': int(graph_csr.nnz),\\n    },\\n    'integrity_assertions': {**source_assertions, **training_assertions, **evaluation_assertions},\\n    'training': {\\n        'epochs': epoch_records,\\n        'wall_seconds': training_wall_seconds,\\n        'peak_gpu_memory_mb': training_peak_gpu_mb,\\n        'negative_draws': train_rows * CONFIG['training']['epochs'],\\n        'negative_resamples': total_negative_resamples,\\n        'optimizer_steps': optimizer_steps,\\n        'deterministic_first_batch_trace_passed': deterministic_replay_passed,\\n    },\\n    'validation': {\\n        'metrics': overall_metrics,\\n        'target_item_cohorts': item_cohort_metrics,\\n        'user_activity_cohorts': user_cohort_metrics,\\n        'recommendation_exposure': {\\n            'recommendation_slots': recommendation_slots,\\n            'unique_items_at_k': int(recommended_mask.sum()),\\n            'catalog_coverage_at_k': coverage,\\n            'item_cohort_share': {label: exposure_counts[label] / recommendation_slots for label in ('head', 'body', 'tail')},\\n        },\\n        'wall_seconds': evaluation_wall_seconds,\\n        'peak_gpu_memory_mb': evaluation_peak_gpu_mb,\\n        'topk_boundary_tie_rows': boundary_tie_rows,\\n    },\\n    'matched_comparison': {\\n        'comparison_boundary': 'M0 uniform and M1 degree-aware are the budget-matched controls for M2. Deltas are descriptive for one fixed validation seed; no significance or test-set claim.',\\n        'uniform_m0': {\\n            'ndcg_at_20': uniform_metrics['ndcg_at_k'],\\n            'recall_at_20': uniform_metrics['recall_at_k'],\\n            'hits_at_20': uniform_metrics['hits_at_k'],\\n            'catalog_coverage_at_20': uniform_m0['validation']['recommendation_exposure']['catalog_coverage_at_k'],\\n            'training_wall_seconds': uniform_m0['training']['wall_seconds'],\\n            'training_peak_gpu_memory_mb': uniform_m0['training']['peak_gpu_memory_mb'],\\n        },\\n        'degree_m1': {\\n            'ndcg_at_20': degree_metrics['ndcg_at_k'],\\n            'recall_at_20': degree_metrics['recall_at_k'],\\n            'hits_at_20': degree_metrics['hits_at_k'],\\n            'catalog_coverage_at_20': degree_m1['validation']['recommendation_exposure']['catalog_coverage_at_k'],\\n            'training_wall_seconds': degree_m1['training']['wall_seconds'],\\n            'training_peak_gpu_memory_mb': degree_m1['training']['peak_gpu_memory_mb'],\\n        },\\n        'frontier_m2': {\\n            'ndcg_at_20': overall_metrics['ndcg_at_k'],\\n            'recall_at_20': overall_metrics['recall_at_k'],\\n            'hits_at_20': overall_metrics['hits_at_k'],\\n            'catalog_coverage_at_20': coverage,\\n            'training_wall_seconds': training_wall_seconds,\\n            'training_peak_gpu_memory_mb': training_peak_gpu_mb,\\n        },\\n        'frontier_minus_uniform': {\\n            'ndcg_at_20': overall_metrics['ndcg_at_k'] - uniform_metrics['ndcg_at_k'],\\n            'recall_at_20': overall_metrics['recall_at_k'] - uniform_metrics['recall_at_k'],\\n            'hits_at_20': overall_metrics['hits_at_k'] - uniform_metrics['hits_at_k'],\\n            'catalog_coverage_at_20': coverage - uniform_m0['validation']['recommendation_exposure']['catalog_coverage_at_k'],\\n            'training_wall_seconds': training_wall_seconds - uniform_m0['training']['wall_seconds'],\\n            'training_peak_gpu_memory_mb': training_peak_gpu_mb - uniform_m0['training']['peak_gpu_memory_mb'],\\n        },\\n        'frontier_minus_degree': {\\n            'ndcg_at_20': overall_metrics['ndcg_at_k'] - degree_metrics['ndcg_at_k'],\\n            'recall_at_20': overall_metrics['recall_at_k'] - degree_metrics['recall_at_k'],\\n            'hits_at_20': overall_metrics['hits_at_k'] - degree_metrics['hits_at_k'],\\n            'catalog_coverage_at_20': coverage - degree_m1['validation']['recommendation_exposure']['catalog_coverage_at_k'],\\n            'training_wall_seconds': training_wall_seconds - degree_m1['training']['wall_seconds'],\\n            'training_peak_gpu_memory_mb': training_peak_gpu_mb - degree_m1['training']['peak_gpu_memory_mb'],\\n        },\\n    },\\n    'reference_comparison': {\\n        'comparison_boundary': 'MostPop, BPR-MF and full LightGCN are context references, not budget-matched controls.',\\n        'mostpop': {'ndcg_at_20': mostpop_metrics['ndcg_at_k'], 'recall_at_20': mostpop_metrics['recall_at_k']},\\n        'bpr_mf': {'ndcg_at_20': bpr_metrics['ndcg_at_k'], 'recall_at_20': bpr_metrics['recall_at_k']},\\n        'full_lightgcn': {'ndcg_at_20': full_metrics['ndcg_at_k'], 'recall_at_20': full_metrics['recall_at_k']},\\n        'uniform_m0': {'ndcg_at_20': uniform_metrics['ndcg_at_k'], 'recall_at_20': uniform_metrics['recall_at_k']},\\n        'degree_m1': {'ndcg_at_20': degree_metrics['ndcg_at_k'], 'recall_at_20': degree_metrics['recall_at_k']},\\n        'frontier_m2': {'ndcg_at_20': overall_metrics['ndcg_at_k'], 'recall_at_20': overall_metrics['recall_at_k']},\\n    },\\n    'environment': {\\n        'python': platform.python_version(), 'platform': platform.platform(),\\n        'torch': torch.__version__, 'cuda': torch.version.cuda,\\n        'gpu': torch.cuda.get_device_name(device), 'process_peak_rss_mb': process_peak_rss_mb(),\\n    },\\n    'claim_boundary': CONFIG['claim_boundary'],\\n}\\nprint(json.dumps({\\n    'status': summary['status'],\\n    'metrics': overall_metrics,\\n    'coverage_at_20': coverage,\\n    'training_wall_seconds': training_wall_seconds,\\n    'training_peak_gpu_mb': training_peak_gpu_mb,\\n    'all_assertions_pass': all(summary['integrity_assertions'].values()),\\n}, ensure_ascii=False, indent=2))\\n\"}}")


In [ ]:
# @title 3. Kiểm protocol và định nghĩa runner
import weakref
def canonical_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n').encode('utf-8')


def semantic_sha256(value):
    return hashlib.sha256(canonical_bytes(value)).hexdigest()


def array_sha256(values):
    values = np.ascontiguousarray(values)
    digest = hashlib.sha256()
    digest.update(canonical_bytes({'shape': list(values.shape), 'dtype': str(values.dtype)}))
    digest.update(memoryview(values).cast('B'))
    return digest.hexdigest()


def embedding_sha256(weight):
    digest = hashlib.sha256()
    digest.update(canonical_bytes({'shape': list(weight.shape), 'dtype': str(weight.dtype)}))
    for start in range(0, weight.shape[0], 65536):
        chunk = weight[start:start + 65536].detach().cpu().contiguous().numpy()
        digest.update(memoryview(chunk).cast('B'))
    return digest.hexdigest()


def validate_frozen_payload():
    if semantic_sha256(PROTOCOL) != REGISTERED_PROTOCOL_SHA256:
        raise ValueError('Protocol đã thay đổi; không chạy hoặc tự sửa checksum.')
    for method in ('M0', 'M1', 'M2'):
        entry = PROTOCOL['methods'][method]
        if semantic_sha256(BASE_CONFIGS[method]) != entry['base_config_semantic_sha256']:
            raise ValueError(f'Config {method} đã thay đổi.')
        for stage in ('training', 'evaluation'):
            actual = hashlib.sha256(ENGINES[method][stage].encode('utf-8')).hexdigest()
            if actual != entry['runner_source_sha256'][stage]:
                raise ValueError(f'Code {method}/{stage} đã thay đổi.')
    for name, source in (('foundation', FOUNDATION_SOURCE), ('data', DATA_SOURCE)):
        if hashlib.sha256(source.encode('utf-8')).hexdigest() != PROTOCOL['shared_source_sha256'][name]:
            raise ValueError(f'Code chuẩn bị {name} đã thay đổi.')


def planned_runs():
    return [
        {'run_id': seed['seed_id'] + '_' + method, 'seed_id': seed['seed_id'],
         'method': method, 'training_seed': seed['training_seed'],
         'sampler_seed': seed['sampler_seed']}
        for seed in PROTOCOL['additional_seeds'] for method in seed['method_order']
    ]


def trial_config(plan):
    config = copy.deepcopy(BASE_CONFIGS[plan['method']])
    config['config_id'] = PROTOCOL['protocol_id'] + '-' + plan['run_id']
    config['training']['seed'] = int(plan['training_seed'])
    config['sampler']['seed'] = int(plan['sampler_seed'])
    config['claim_boundary'] = PROTOCOL['claim_boundary']
    return config


def expected_identity(plan):
    return {
        'run_id': plan['run_id'], 'protocol_sha256': REGISTERED_PROTOCOL_SHA256,
        'runtime_config_sha256': semantic_sha256(trial_config(plan)),
        'train_sha256': train_sha, 'validation_sha256': validation_sha,
        'target_order_sha256': TARGET_ORDER_SHA256, 'validation_rows': int(validation_rows),
        'runtime_environment_sha256': RUNTIME_ENVIRONMENT_SHA256,
    }


def check_pairing(summaries):
    if not summaries:
        raise ValueError('Không có run để kiểm pairing.')
    first = summaries[0]
    identity_keys = ('protocol_sha256', 'train_sha256', 'validation_sha256',
                     'target_order_sha256', 'validation_rows')
    for summary in summaries[1:]:
        if any(summary[key] != first[key] for key in identity_keys):
            raise ValueError('Ba method không dùng cùng data/target order/protocol.')
        if summary['pairing'] != first['pairing']:
            raise ValueError('Embedding initialization, pair order hoặc negative draw không paired.')
        for key in ('model', 'training', 'evaluation'):
            if summary['registered_config'][key] != first['registered_config'][key]:
                raise ValueError(f'Matched {key} contract không bằng nhau.')
        for key in ('gpu', 'python', 'torch', 'cuda'):
            if summary['environment'][key] != first['environment'][key]:
                raise ValueError(f'Môi trường {key} không bằng nhau trong cùng paired seed.')


def compare_rank_vectors(lhs, rhs, labels, k):
    lhs, rhs, labels = np.asarray(lhs), np.asarray(rhs), np.asarray(labels)
    if lhs.ndim != 1 or lhs.shape != rhs.shape or lhs.shape != labels.shape:
        raise ValueError('Rank vectors/labels không cùng target order và kích thước.')
    result = {}
    groups = [('overall', np.ones(lhs.size, dtype=bool))]
    groups += [(str(label), labels == label) for label in sorted(set(labels.tolist()))]
    for label, mask in groups:
        a, b = lhs[mask], rhs[mask]
        ah, bh = a <= k, b <= k
        gained, lost = int((ah & ~bh).sum()), int((~ah & bh).sum())
        result[label] = {
            'rows': int(mask.sum()), 'gained_hits': gained, 'lost_hits': lost,
            'net_hits': gained - lost, 'both_hit_rows': int((ah & bh).sum()),
            'neither_hit_rows': int((~ah & ~bh).sum()),
            'rank_improved_rows': int((a < b).sum()),
            'rank_worsened_rows': int((a > b).sum()), 'same_rank_rows': int((a == b).sum()),
        }
    return result


def rank_diagnostics(ranks, labels):
    ranks, labels = np.asarray(ranks), np.asarray(labels)
    if ranks.ndim != 1 or ranks.shape != labels.shape or np.any(ranks < 1):
        raise ValueError('Rank diagnostics cần positive ranks và aligned labels.')
    result = {}
    for label in sorted(set(labels.tolist())):
        values = ranks[labels == label]
        result[str(label)] = {
            'rows': int(values.size), 'hits_at_20': int((values <= 20).sum()),
            'share_at_100': float((values <= 100).mean()),
            'share_at_1000': float((values <= 1000).mean()),
            'median_rank': float(np.median(values)),
            'quartile_25_rank': float(np.percentile(values, 25)),
            'quartile_75_rank': float(np.percentile(values, 75)),
        }
    return result


def safe_child(root, relative):
    relative = Path(relative)
    if relative.is_absolute() or '..' in relative.parts:
        raise ValueError('Completion marker có path ngoài run folder.')
    child = (root / relative).resolve()
    if not child.is_relative_to(root.resolve()):
        raise ValueError('Completion marker trỏ ngoài run folder.')
    return child


def validate_completed_summary(summary, ranks, identity):
    if any(summary.get(key) != value for key, value in identity.items()):
        raise ValueError('Completed run không khớp protocol/config/data/target order hiện tại.')
    if semantic_sha256(summary['registered_config']) != identity['runtime_config_sha256']:
        raise ValueError('Runtime config fingerprint không hợp lệ.')
    gates = summary.get('integrity_assertions', {})
    if not gates or not all(value is True for value in gates.values()):
        raise ValueError('Completed run có integrity gate fail, rỗng hoặc không phải boolean.')
    if summary['training']['optimizer_steps'] != 300 or len(summary['training']['epochs']) != 5:
        raise ValueError('Completed run chưa đủ 5 epoch/300 optimizer step.')
    ranks = np.asarray(ranks)
    if ranks.ndim != 1 or ranks.size != identity['validation_rows'] or ranks.dtype.kind not in 'iu':
        raise ValueError('Rank artifact sai kích thước/dtype.')
    if np.any(ranks < 1):
        raise ValueError('Rank artifact có rank không hợp lệ.')
    metrics = summary['validation']['metrics']
    if metrics['rows'] != ranks.size or metrics['hits_at_k'] != int((ranks <= 20).sum()):
        raise ValueError('Summary hit/row count không khớp rank artifact.')
    if metrics.get('k') != 20 or ranks.size == 0:
        raise ValueError('Evaluator phải dùng k=20 và có target.')
    hits = ranks <= 20
    discounts = np.zeros(ranks.size, dtype=np.float64)
    discounts[hits] = 1.0 / np.log2(ranks[hits] + 1.0)
    if (not np.isfinite(metrics['ndcg_at_k']) or not np.isfinite(metrics['recall_at_k'])
            or abs(metrics['recall_at_k'] - float(hits.mean())) > 1e-12
            or abs(metrics['ndcg_at_k'] - float(discounts.mean())) > 1e-12):
        raise ValueError('Validation NDCG/Recall không khớp exact rank artifact.')
    inputs = summary['pairing']['epoch_inputs']
    if [entry['epoch'] for entry in inputs] != [1, 2, 3, 4, 5]:
        raise ValueError('Thiếu paired input fingerprints của epoch.')


def load_completed_run(run_dir, identity):
    marker_path = run_dir / 'COMPLETED.json'
    if not marker_path.exists():
        return None
    try:
        marker = json.loads(marker_path.read_text(encoding='utf-8'))
        paths = {key: safe_child(run_dir, marker[key]['path']) for key in ('summary', 'ranks')}
        for key, path in paths.items():
            if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != marker[key]['sha256']:
                raise ValueError(f'Completed {key} thiếu hoặc checksum sai; không tự overwrite.')
        summary = json.loads(paths['summary'].read_text(encoding='utf-8'))
        with np.load(paths['ranks'], allow_pickle=False) as payload:
            ranks = payload['ranks'].copy()
        validate_completed_summary(summary, ranks, identity)
        return {'summary': summary, 'ranks': ranks, 'paths': paths}
    except (OSError, KeyError, TypeError, json.JSONDecodeError) as error:
        raise ValueError('Completion marker/artifact không hợp lệ; gửi lỗi cho người hướng dẫn.') from error


def save_completed_run(run_dir, summary, ranks):
    if (run_dir / 'COMPLETED.json').exists():
        raise FileExistsError('Run đã complete; không được overwrite.')
    identity_keys = ('run_id', 'protocol_sha256', 'runtime_config_sha256', 'train_sha256',
                     'validation_sha256', 'target_order_sha256', 'validation_rows',
                     'runtime_environment_sha256')
    identity = {key: summary[key] for key in identity_keys}
    validate_completed_summary(summary, ranks, identity)
    attempt = run_dir / 'attempts' / uuid.uuid4().hex
    attempt.mkdir(parents=True, exist_ok=False)
    ranks_path = attempt / 'validation_ranks.npz'
    with ranks_path.open('xb') as handle:
        np.savez_compressed(handle, ranks=np.asarray(ranks, dtype=np.int32))
    summary_path = attempt / 'summary.json'
    with summary_path.open('xb') as handle:
        handle.write(canonical_bytes(summary))
    marker = {key: {'path': str(path.relative_to(run_dir)),
                    'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
              for key, path in (('summary', summary_path), ('ranks', ranks_path))}
    # Close the full marker in this attempt before publication. Never publish a partial write.
    staged_marker = attempt / 'completion_ready.json'
    with staged_marker.open('xb') as handle:
        handle.write(canonical_bytes(marker))
    if json.loads(staged_marker.read_text(encoding='utf-8')) != marker:
        raise ValueError('Staged completion marker integrity fail.')
    marker_path = run_dir / 'COMPLETED.json'
    if marker_path.exists():
        raise FileExistsError('Run đã complete; không overwrite marker.')
    # Same-folder-filesystem rename publishes a closed marker. One runtime per output folder.
    staged_marker.rename(marker_path)
    return load_completed_run(run_dir, identity)


def measure_row(summary):
    metrics = summary['validation']['metrics']
    return {
        'ndcg_at_20': float(metrics['ndcg_at_k']),
        'recall_at_20': float(metrics['recall_at_k']), 'hits_at_20': int(metrics['hits_at_k']),
        'catalog_coverage_at_20': float(summary['validation']['recommendation_exposure']['catalog_coverage_at_k']),
        'training_wall_seconds': float(summary['training']['wall_seconds']),
        'training_peak_gpu_memory_mb': float(summary['training']['peak_gpu_memory_mb']),
        'sampler_seconds': float(sum(epoch['sampler_seconds'] for epoch in summary['training']['epochs'])),
    }


def build_aggregate(completed, legacy, labels):
    plans = planned_runs()
    comparisons = []
    for seed in PROTOCOL['additional_seeds']:
        ids = {method: seed['seed_id'] + '_' + method for method in ('M0', 'M1', 'M2')}
        if not all(run_id in completed for run_id in ids.values()):
            continue
        summaries = [completed[ids[method]]['summary'] for method in ('M0', 'M1', 'M2')]
        check_pairing(summaries)
        measures = {method: measure_row(completed[run_id]['summary']) for method, run_id in ids.items()}
        comparisons.append({
            'seed_id': seed['seed_id'], 'training_seed': seed['training_seed'],
            'measures': measures, 'pairing_verified': True,
            'm2_minus_m1': {key: measures['M2'][key] - measures['M1'][key] for key in measures['M2']},
            'm2_minus_m0': {key: measures['M2'][key] - measures['M0'][key] for key in measures['M2']},
            'm2_vs_m1_target_transitions': compare_rank_vectors(
                completed[ids['M2']]['ranks'], completed[ids['M1']]['ranks'], labels, 20),
            'm2_vs_m0_target_transitions': compare_rank_vectors(
                completed[ids['M2']]['ranks'], completed[ids['M0']]['ranks'], labels, 20),
        })
    legacy_measures = {method: measure_row(summary) for method, summary in legacy.items()}
    environment_keys = ('gpu', 'python', 'torch', 'cuda')
    legacy_compatible = all(
        item['summary']['environment'][key] == legacy[method]['environment'][key]
        for item in completed.values() for method in ('M0', 'M1', 'M2') for key in environment_keys
    )
    quality_statistics = {}
    for method in ('M0', 'M1', 'M2'):
        method_rows = ([legacy_measures[method]] if legacy_compatible else []) + [c['measures'][method] for c in comparisons]
        quality_statistics[method] = {
            key: {'seed_count': len(method_rows), 'mean': float(np.mean([r[key] for r in method_rows])) if method_rows else None,
                  'sample_sd': float(np.std([r[key] for r in method_rows], ddof=1)) if len(method_rows) > 1 else None}
            for key in ('ndcg_at_20', 'recall_at_20', 'catalog_coverage_at_20')
        }
    quality_delta_rows = [{
        key: legacy_measures['M2'][key] - legacy_measures['M1'][key]
        for key in ('ndcg_at_20', 'recall_at_20')
    }] + [c['m2_minus_m1'] for c in comparisons]
    return {
        'status': 'PAIRED_VALIDATION_COMPLETE' if len(completed) == len(plans) else 'PAIRED_VALIDATION_PARTIAL',
        'protocol_sha256': REGISTERED_PROTOCOL_SHA256, 'completed_runs': len(completed),
        'planned_runs': len(plans), 'pending_run_ids': [p['run_id'] for p in plans if p['run_id'] not in completed],
        'additional_seed_comparisons': comparisons,
        'legacy_seed_context': {'seed_id': 's0', 'training_seed': 20260913, 'measures': legacy_measures,
                                'summary_sha256': PROTOCOL['legacy_summary_sha256'],
                                'row_level_ranks_available': False,
                                'environments': {m: s['environment'] for m, s in legacy.items()}},
        'quality_statistics': quality_statistics,
        'quality_statistics_include_legacy_seed': legacy_compatible,
        'quality_statistics_seed_ids': (['s0'] if legacy_compatible else []) + [c['seed_id'] for c in comparisons],
        'm2_minus_m1_quality_signs_including_legacy_seed': {
            key: [int(np.sign(row[key])) for row in quality_delta_rows]
            for key in ('ndcg_at_20', 'recall_at_20')
        },
        'run_summaries': {run_id: item['summary'] for run_id, item in completed.items()},
        'claim_boundary': PROTOCOL['claim_boundary'],
        'resource_boundary': PROTOCOL['resource_boundary'],
        'test_targets_read': False, 'automatic_promotion_performed': False,
    }


def render_report(aggregate):
    lines = ['# Paired sampling validation', '',
             'Trạng thái: ' + aggregate['status'], '',
             f"Hoàn thành {aggregate['completed_runs']}/{aggregate['planned_runs']} run mới.", '',
             '| Run | NDCG@20 | Recall@20 | Train (s) | Peak GPU (MB) |',
             '|---|---:|---:|---:|---:|']
    for run_id, summary in aggregate['run_summaries'].items():
        row = measure_row(summary)
        lines.append(f"| {run_id} | {row['ndcg_at_20']:.8f} | {row['recall_at_20']:.8f} | "
                     f"{row['training_wall_seconds']:.2f} | {row['training_peak_gpu_memory_mb']:.2f} |")
    lines += ['', '## Paired M2 − M1', '',
              '| Seed | ΔNDCG | ΔRecall | ΔTrain (s) | Pairing |',
              '|---|---:|---:|---:|---|']
    for comparison in aggregate['additional_seed_comparisons']:
        delta = comparison['m2_minus_m1']
        lines.append(f"| {comparison['seed_id']} | {delta['ndcg_at_20']:+.8f} | "
                     f"{delta['recall_at_20']:+.8f} | {delta['training_wall_seconds']:+.2f} | pass |")
    lines += ['', '## Giới hạn', '', aggregate['claim_boundary'], '', aggregate['resource_boundary'], '',
              'JSON chứa rank transitions head/body/tail; quality chỉ gộp s0 khi các runtime field đã ghi khớp.',
              'Seed s0 không có row-level rank artifact, nên không suy diễn target overlap cho s0.',
              'Không chọn best seed, không p-value/significance claim, không đọc test và không auto-promote M2.']
    return '\n'.join(lines) + '\n'


def export_bundle(output_dir, completed, aggregate, report):
    export_dir = output_dir / 'exports' / uuid.uuid4().hex
    export_dir.mkdir(parents=True, exist_ok=False)
    archive = export_dir / 'paired_sampling_validation_bundle.zip'
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
        bundle.writestr('paired_sampling_validation_protocol.json', canonical_bytes(PROTOCOL))
        bundle.writestr('paired_sampling_validation_summary.json', canonical_bytes(aggregate))
        bundle.writestr('PAIRED_SAMPLING_VALIDATION_vn.md', report)
        for run_id, item in completed.items():
            for key, filename in (('summary', 'summary.json'), ('ranks', 'validation_ranks.npz')):
                bundle.write(item['paths'][key], 'runs/' + run_id + '/' + filename)
    with zipfile.ZipFile(archive) as bundle:
        if bundle.testzip() is not None:
            raise ValueError('Export ZIP integrity fail.')
    return archive


def cuda_tensor_refs():
    # Weak references only: the diagnostic must not keep model tensors alive.
    # This enumerates GC-visible Python tensors, not native CUDA workspaces.
    refs = []
    for value in gc.get_objects():
        if issubclass(type(value), torch.Tensor) and value.is_cuda:
            refs.append((weakref.ref(value), {
                'shape': list(value.shape), 'dtype': str(value.dtype), 'device': str(value.device),
            }))
    return refs


def cuda_memory_state():
    refs = cuda_tensor_refs()
    return {
        'allocated_mb': torch.cuda.memory_allocated() / (1024 ** 2),
        'reserved_mb': torch.cuda.memory_reserved() / (1024 ** 2),
        'live_cuda_tensor_count': len(refs),
        'tensor_examples': [metadata for _, metadata in refs[:10]],
    }


def run_trial(plan):
    config = trial_config(plan)
    namespace = dict(SHARED_DATA)
    namespace.update({
        '__name__': '__main__', 'CONFIG': config,
        'array_sha256': array_sha256, 'embedding_sha256': embedding_sha256,
    })
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()
    before = cuda_memory_state()
    if before['live_cuda_tensor_count']:
        raise RuntimeError('Có CUDA tensor thật còn sống từ lượt khác: ' + json.dumps(before, ensure_ascii=False)
                           + '. Restart runtime, không xóa output đã complete.')
    # allocated may include persistent cuBLAS workspace even with no Python tensors.
    # Do not clear private workspaces, change the engines, or subtract from gross peaks.
    finished = False
    started_at = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    try:
        exec(compile(ENGINES[plan['method']]['training'], plan['run_id'] + '-training', 'exec'), namespace)
        exec(compile(ENGINES[plan['method']]['evaluation'], plan['run_id'] + '-validation', 'exec'), namespace)
        summary = namespace['summary']
        # Old reference deltas are not paired with this new seed.
        summary.pop('matched_comparison', None)
        summary.pop('reference_comparison', None)
        summary.update(expected_identity(plan))
        summary['status'] = 'PAIRED_SAMPLING_RUN_COMPLETE'
        summary['seed_id'] = plan['seed_id']
        summary['method'] = plan['method']
        summary['started_at_utc'] = started_at
        summary['pairing'] = {
            'initial_embedding_sha256': namespace['initial_embedding_sha256'],
            'epoch_inputs': namespace['epoch_input_fingerprints'],
        }
        summary['validation']['target_rank_diagnostics'] = rank_diagnostics(namespace['ranks'], TARGET_ITEM_LABELS)
        summary['environment']['numpy'] = np.__version__
        summary['environment']['scipy'] = scipy.__version__
        summary['training']['timing_boundary'] = PROTOCOL['resource_boundary']
        ranks = namespace['ranks'].copy()
        finished = True
        return summary, ranks
    finally:
        try:
            tracked_refs = cuda_tensor_refs()
            namespace.clear()
            gc.collect()
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            after = cuda_memory_state()
            surviving_refs = sum(ref() is not None for ref, _ in tracked_refs)
            if finished:
                if surviving_refs or after['live_cuda_tensor_count']:
                    raise RuntimeError('CUDA tensor của lượt vừa chạy chưa được giải phóng: '
                                       + json.dumps(after, ensure_ascii=False)
                                       + '. Dừng trước khi lưu; không xóa các run complete trước đó.')
                summary['cuda_cleanup'] = {
                    'runner_revision': 'cuda-workspace-ownership-hotfix-v1',
                    'before': before, 'after': after,
                    'tracked_trial_tensors': len(tracked_refs),
                    'tracked_trial_tensors_still_alive': surviving_refs,
                    'boundary': 'GC-visible Python CUDA tensors released; native workspace may remain. '
                                'Training peak remains gross, unchanged.',
                }
                summary['integrity_assertions']['runner_cuda_tensors_released'] = True
                print(f"CUDA_CLEANUP_OK: allocated={after['allocated_mb']:.3f} MiB; "
                      f"live_cuda_tensors={after['live_cuda_tensor_count']}")
        except Exception as cleanup_error:
            # A CUDA error/traceback can retain engine tensors. Preserve its real cause.
            namespace.clear()
            if finished:
                raise
            print('CUDA_CLEANUP_AFTER_FAILURE:', type(cleanup_error).__name__, str(cleanup_error))


def collect_completed():
    completed = {}
    for plan in planned_runs():
        item = load_completed_run(OUTPUT_DIR / 'runs' / plan['run_id'], expected_identity(plan))
        if item is not None:
            completed[plan['run_id']] = item
    return completed

validate_frozen_payload()
print('PROTOCOL_PAYLOAD_VERIFIED:', REGISTERED_PROTOCOL_SHA256)
for plan in planned_runs():
    print(plan)


## Kiểm dữ liệu một lần

Cell này đọc training/validation artifact, kiểm checksum và dựng strict-prior history/CSR dùng chung. Chờ `SOURCE_GATE_PASSED` trước khi chạy training. Nếu thiếu file, gửi nguyên dòng lỗi; không chạy lại data pipeline để chữa bằng một dataset khác.


In [ ]:
# @title 4. Preflight dữ liệu và GPU — chờ SOURCE_GATE_PASSED
import gc
import time
import platform
import torch
import scipy

validate_frozen_payload()
if not torch.cuda.is_available():
    raise RuntimeError('Cần GPU CUDA/T4. Đổi runtime rồi chạy lại từ đầu.')
if torch.cuda.get_device_name(0) != PROTOCOL['required_gpu']:
    raise RuntimeError(f"Cần {PROTOCOL['required_gpu']}, đang có {torch.cuda.get_device_name(0)}. Không tự đổi budget.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
protocol_path = OUTPUT_DIR / 'registered_protocol.json'
if protocol_path.exists():
    if protocol_path.read_bytes() != canonical_bytes(PROTOCOL):
        raise ValueError('Output folder thuộc protocol khác hoặc protocol file hỏng; không overwrite.')
else:
    with protocol_path.open('xb') as handle:
        handle.write(canonical_bytes(PROTOCOL))
print('PROTOCOL_REGISTERED_ON_DRIVE:', protocol_path)
print('GPU:', torch.cuda.get_device_name(0))
print('Python/Torch/CUDA:', platform.python_version(), torch.__version__, torch.version.cuda)
RUNTIME_ENVIRONMENT_SHA256 = semantic_sha256({
    'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0), 'numpy': np.__version__, 'scipy': scipy.__version__,
})
print('Runtime environment SHA-256:', RUNTIME_ENVIRONMENT_SHA256)

SHARED_DATA = {
    '__name__': '__main__', 'Path': Path, 'CONFIG': copy.deepcopy(BASE_CONFIGS['M2']),
    'GRAPH_DIR': GRAPH_DIR, 'MANIFEST_PATH': MANIFEST_PATH,
    'MOSTPOP_PATH': MOSTPOP_PATH, 'BPR_PATH': BPR_PATH,
    'FULL_LIGHTGCN_PATH': FULL_LIGHTGCN_PATH,
    'UNIFORM_PATH': UNIFORM_PATH, 'DEGREE_PATH': DEGREE_PATH,
}
exec(compile(FOUNDATION_SOURCE, 'frozen-foundation', 'exec'), SHARED_DATA)
if SHARED_DATA['sha256_file'](FRONTIER_PATH) != PROTOCOL['legacy_summary_sha256']['M2']:
    raise ValueError('M2 s0 summary checksum không khớp; không chạy thêm.')
frontier_s0 = json.loads(FRONTIER_PATH.read_text(encoding='utf-8'))
if not frontier_s0['integrity_assertions'] or not all(v is True for v in frontier_s0['integrity_assertions'].values()):
    raise ValueError('M2 s0 integrity gate fail.')
exec(compile(DATA_SOURCE, 'frozen-data-and-history', 'exec'), SHARED_DATA)
LEGACY_RESULTS = {
    'M0': SHARED_DATA['uniform_m0'], 'M1': SHARED_DATA['degree_m1'], 'M2': frontier_s0,
}
for method, old in LEGACY_RESULTS.items():
    for key, artifact_key in (('train', 'train_edges'), ('validation', 'validation_targets')):
        if old['source'][artifact_key]['sha256'] != PROTOCOL['source_hashes'][key]:
            raise ValueError(f'Data checksum khác giữa các legacy method: {method}.')
    old_config = {k: v for k, v in old['registered_config'].items() if k != 'file_sha256'}
    if old_config != BASE_CONFIGS[method]:
        raise ValueError(f'Legacy {method} config khác config đóng băng.')
train_sha = SHARED_DATA['train_sha']
validation_sha = SHARED_DATA['validation_sha']
if train_sha != PROTOCOL['source_hashes']['train'] or validation_sha != PROTOCOL['source_hashes']['validation']:
    raise ValueError('Data hiện tại khác artifact đã dùng ở s0.')
validation_rows = SHARED_DATA['validation_rows']
if SHARED_DATA['math'].ceil(SHARED_DATA['train_rows'] / 65536) * 5 != 300:
    raise ValueError('Data/batch không tạo đúng 300 optimizer step; không tự sửa config.')
TARGET_ORDER_SHA256 = semantic_sha256({
    name: array_sha256(SHARED_DATA[name])
    for name in ('target_users', 'target_items', 'target_timestamps', 'target_source_rows', 'target_candidate_counts')
})
TARGET_ITEM_LABELS = np.asarray([
    SHARED_DATA['item_cohort'](SHARED_DATA['item_degrees'][item])
    for item in SHARED_DATA['target_items']
])
for key in ('csr_rows', 'csr_cols', 'csr_data', 'edge_ids', 'global_items',
            'base_history', 'targets_by_user', 'prior', 'eval_user_mask',
            'preflight_indices', 'preflight_users', 'preflight_negatives'):
    SHARED_DATA.pop(key, None)
gc.collect()
COMPLETED_RUNS = collect_completed()
print('SOURCE_GATE_PASSED')
print('Train/validation rows:', SHARED_DATA['train_rows'], validation_rows)
print('Target order SHA-256:', TARGET_ORDER_SHA256)
print('Đã complete và kiểm checksum:', list(COMPLETED_RUNS))
print('Số lượt còn lại:', len(planned_runs()) - len(COMPLETED_RUNS))


## Chạy hai nhóm paired

`s1`: M0 → M1 → M2. `s2`: M2 → M0 → M1. Mỗi lượt 5 epoch, đúng 300 optimizer step. Chờ từng dòng `COMPLETED_AND_SAVED`; việc lưu completion marker chỉ diễn ra sau khi summary/ranks đã ghi và kiểm checksum.

Thời gian training tham khảo từ s0 khoảng 80 phút cho sáu lượt, cộng preflight/validation và overhead Colab. Không coi đây là cam kết thời gian. Cell có lỗi thì không tự giảm batch hay lấy seed khác.


In [ ]:
# @title 5. Chạy sáu lượt đã khóa — KHÔNG chạy notebook này đồng thời ở hai runtime
validate_frozen_payload()
COMPLETED_RUNS = collect_completed()
for position, plan in enumerate(planned_runs(), start=1):
    run_id = plan['run_id']
    if run_id in COMPLETED_RUNS:
        print(f'[{position}/6] SKIP_VERIFIED {run_id}')
        continue
    print(f"\n[{position}/6] START {run_id} — training seed {plan['training_seed']}")
    try:
        trial_summary, trial_ranks = run_trial(plan)
        # Before saving, compare actual inputs/environment with any sibling already complete.
        siblings = [item['summary'] for old_id, item in COMPLETED_RUNS.items()
                    if old_id.startswith(plan['seed_id'] + '_')]
        check_pairing(siblings + [trial_summary])
        run_dir = OUTPUT_DIR / 'runs' / run_id
        COMPLETED_RUNS[run_id] = save_completed_run(run_dir, trial_summary, trial_ranks)
        print(f'[{position}/6] COMPLETED_AND_SAVED {run_id}')
        print('NDCG/Recall:', trial_summary['validation']['metrics']['ndcg_at_k'],
              trial_summary['validation']['metrics']['recall_at_k'])
        del trial_summary, trial_ranks
        gc.collect()
    except BaseException:
        print('Lượt đang chạy chưa complete. Các lượt complete trước đó vẫn ở Drive.')
        print('Không xóa output hoặc sửa seed/budget. Có thể chạy cell export để gửi ZIP partial.')
        raise
print('ALL_6_RUNS_SAVED')


## Xuất kết quả

ZIP complete phải có `PAIRED_VALIDATION_COMPLETE` và 6/6 run mới. Nếu Colab lỗi giữa chừng, cell export vẫn có thể xuất `PAIRED_VALIDATION_PARTIAL`; gửi ZIP partial kèm lỗi. Chỉ export artifact được whitelist, không checkpoint/raw data. Mỗi lần export dùng folder mới nên không ghi đè ZIP cũ.


In [ ]:
# @title 6. Export ZIP — cũng dùng được khi mới chạy được một phần
COMPLETED_RUNS = collect_completed()
AGGREGATE = build_aggregate(COMPLETED_RUNS, LEGACY_RESULTS, TARGET_ITEM_LABELS)
REPORT = render_report(AGGREGATE)
BUNDLE_PATH = export_bundle(OUTPUT_DIR, COMPLETED_RUNS, AGGREGATE, REPORT)
print(REPORT)
print('BUNDLE_PATH:', BUNDLE_PATH)
print('BUNDLE_SHA256:', hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest())
print('Gửi file ZIP này, không cần gửi checkpoint hoặc raw data.')


In [ ]:
# @title 7. Tải ZIP về máy
from google.colab import files
if 'BUNDLE_PATH' not in globals() or not BUNDLE_PATH.is_file():
    raise RuntimeError('Chạy cell export trước.')
files.download(str(BUNDLE_PATH))
